## Synthetic data generation

Generate Gaussian random fields over a grid of covariance **models**, **ranges**, **sills**, **nugget fractions**, and **shape parameters** (nu / alpha) via exact **circulant embedding**, written as tagged GeoTIFFs. Set `GENERATE = True` in the run cell to write the dataset (resumable: existing files are skipped).

In [1]:
import os, math, hashlib, json, numpy as np, gstools as gs, rasterio
from rasterio.transform import from_origin

# model catalog: shape parameter (None|'nu'|'alpha') -> values to sweep; finite-range flag
GEN_MODELS = {
    "Gaussian":       {"param": None,    "values": [None],               "finite": False},
    "Exponential":    {"param": None,    "values": [None],               "finite": False},
    "Spherical":      {"param": None,    "values": [None],               "finite": True},
    "Cubic":          {"param": None,    "values": [None],               "finite": True},
    "Circular":       {"param": None,    "values": [None],               "finite": True},
    "HyperSpherical": {"param": None,    "values": [None],               "finite": True},
    "Matern":         {"param": "nu",    "values": [0.5, 1.5, 2.5, 5.0], "finite": False},
    "Stable":         {"param": "alpha", "values": [0.5, 1.0, 1.5, 2.0], "finite": False},
    "SuperSpherical": {"param": "nu",    "values": [0.5, 1.5, 4.0],      "finite": True},
    "Rational":       {"param": "alpha", "values": [0.5, 1.0, 3.0],      "finite": False},
    "Integral":       {"param": "nu",    "values": [0.5, 1.5, 4.0],      "finite": False},
    "JBessel":        {"param": "nu",    "values": [1.0, 2.0, 4.0],      "finite": False},
}
GEN_FINITE_RANGE = {m for m, c in GEN_MODELS.items() if c["finite"]}

# parameter grids
GEN_RANGES = [5, 10, 50, 100, 250, 500, 1000, 2000]   # effective ranges
GEN_SILLS  = [0.001, 0.01,  0.1, 0.5, 1, 10]     # TOTAL sill (= field variance)
GEN_NUGGET_FRACS = [round(0.1 * i, 1) for i in range(11)]       # 0.0 .. 1.0 (fraction of total sill)
GEN_PERCENTILE = 0.95

# domain / output
GEN_SHAPE  = (1000, 1000)          # (ny, nx)
GEN_RES    = 1.0
GEN_ORIGIN = (500000.0, 5000000.0)
GEN_CRS, GEN_DTYPE, GEN_SEED = "EPSG:32633", "float32", 42
GEN_OUTDIR = "synthetic_benchmark/grf_dataset/"

In [2]:
def gen_len_scale(cls, target_range, percentile=GEN_PERCENTILE, **kw):
    """Effective range -> gstools len_scale (exact for finite-range models, 95% practical for asymptotic)."""
    if cls.__name__ in GEN_FINITE_RANGE:
        return float(target_range)
    return float(target_range) / cls(dim=2, var=1.0, len_scale=1.0, **kw).percentile_scale(percentile)

def gen_pad(rng, dom=max(GEN_SHAPE), res=GEN_RES):
    """Circulant padding: embed the range, capped at 4. Ranges too large to embed get clamped
    eigenvalues -> the field deteriorates (the intended large-range breakdown)."""
    return min(4, max(2, math.ceil((dom + 3 * rng / res) / dom)))

def grf_circulant(model, shape, res=1.0, seed=None, pad=2):
    """Exact stationary GRF via circulant embedding (Dietrich & Newsam, 1993)."""
    ny, nx = shape; My, Mx = pad * ny, pad * nx
    ky = np.concatenate([np.arange(0, My // 2 + 1), np.arange(My // 2 + 1 - My, 0)])
    kx = np.concatenate([np.arange(0, Mx // 2 + 1), np.arange(Mx // 2 + 1 - Mx, 0)])
    Y, X = np.meshgrid(ky * res, kx * res, indexing="ij")
    lam = np.fft.fft2(model.covariance(np.hypot(X, Y))).real
    lam[lam < 0] = 0.0                                  # clamp (only bites when range too big to embed)
    r = np.random.default_rng(seed)
    xi = r.normal(size=(My, Mx)) + 1j * r.normal(size=(My, Mx))
    return (np.fft.fft2(xi * np.sqrt(lam)) / np.sqrt(My * Mx)).real[:ny, :nx]

def _fmt(x):
    """Filename-safe number: decimal point -> '-'  (0.001 -> '0-001', 1.5 -> '1-5')."""
    return f"{x:g}".replace(".", "-")

def _seed_for(name):
    """Deterministic per-field seed from the filename (stable across kernel restarts)."""
    return int(hashlib.md5(name.encode()).hexdigest()[:8], 16)

def _write(out, data, tags):
    ny, nx = data.shape; west, north = GEN_ORIGIN
    os.makedirs(GEN_OUTDIR, exist_ok=True)
    with rasterio.open(out, "w", driver="GTiff", height=ny, width=nx, count=1, dtype=GEN_DTYPE,
                       crs=GEN_CRS, transform=from_origin(west, north, GEN_RES, GEN_RES),
                       compress="deflate") as dst:
        dst.write(data.astype(GEN_DTYPE), 1)
        dst.update_tags(**{k: ("" if v is None else v) for k, v in tags.items()})

def gen_one(model_name, shape_val, rng_eff, total_sill, nug_frac):
    """Generate one structured-GRF + nugget raster; returns (path, 'written'|'skipped')."""
    cls = getattr(gs, model_name); param = GEN_MODELS[model_name]["param"]
    kw = {param: shape_val} if (param and shape_val is not None) else {}
    nug_pct = int(round(nug_frac * 100))
    tag = (f"_nu{_fmt(shape_val)}" if param == "nu" else
           f"_a{_fmt(shape_val)}" if param == "alpha" else "")
    ny, nx = GEN_SHAPE
    fname = f"{model_name}_r{_fmt(rng_eff)}_s{_fmt(total_sill)}_n{nug_pct}{tag}_{ny}x{nx}.tif"
    out = os.path.join(GEN_OUTDIR, fname)
    if os.path.exists(out):
        return out, "skipped"
    partial, nugget = (1.0 - nug_frac) * total_sill, nug_frac * total_sill
    sd = _seed_for(fname)
    if partial > 0:
        ls = gen_len_scale(cls, rng_eff, **kw)
        field = grf_circulant(cls(dim=2, var=partial, len_scale=ls, **kw),
                              GEN_SHAPE, GEN_RES, seed=sd, pad=gen_pad(rng_eff))
    else:
        ls, field = float("nan"), np.zeros(GEN_SHAPE)
    if nugget > 0:
        field = field + np.sqrt(nugget) * np.random.default_rng(sd + 1).normal(size=GEN_SHAPE)
    _write(out, field, dict(
        variogram_model=model_name, sill=total_sill, partial_sill=partial, nugget=nugget,
        nugget_frac=nug_frac, range=rng_eff, len_scale=ls, range_percentile=GEN_PERCENTILE,
        nu=(shape_val if param == "nu" else None), alpha=(shape_val if param == "alpha" else None),
        generator="circulant_embedding", pad=gen_pad(rng_eff), seed=sd, shape=f"{ny}x{nx}"))
    return out, "written"

def gen_noise(total_sill):
    """Pure-nugget (100%) white-noise field -- one per sill, model/range independent."""
    ny, nx = GEN_SHAPE
    fname = f"Nugget_s{_fmt(total_sill)}_n100_{ny}x{nx}.tif"
    out = os.path.join(GEN_OUTDIR, fname)
    if os.path.exists(out):
        return out, "skipped"
    sd = _seed_for(fname)
    field = np.sqrt(total_sill) * np.random.default_rng(sd).normal(size=GEN_SHAPE)
    _write(out, field, dict(
        variogram_model="Nugget", sill=total_sill, partial_sill=0.0, nugget=total_sill,
        nugget_frac=1.0, range=0.0, len_scale=None, range_percentile=GEN_PERCENTILE,
        nu=None, alpha=None, generator="white_noise", pad=None, seed=sd, shape=f"{ny}x{nx}"))
    return out, "written"

In [3]:
def gen_build_jobs():
    structured = [(m, v, r, s, f)
                  for m, c in GEN_MODELS.items() for v in c["values"]
                  for r in GEN_RANGES for s in GEN_SILLS for f in GEN_NUGGET_FRACS if f < 1.0]
    noise = list(GEN_SILLS)                           # 100% nugget: one per sill
    return structured, noise

_structured, _noise = gen_build_jobs()
_total = len(_structured) + len(_noise)
print(f"model x shape combos : {sum(len(c['values']) for c in GEN_MODELS.values())}")
print(f"structured fields    : {len(_structured):,}")
print(f"pure-noise fields    : {len(_noise)}")
print(f"TOTAL                : {_total:,}   (~{_total*3.5/1000:.0f} GB est. at {GEN_SHAPE} {GEN_DTYPE})")
print(f"output dir           : {GEN_OUTDIR}")

GENERATE = True     # <-- set True to write the dataset (resumable: skips existing files)

if GENERATE:
    written = skipped = 0
    for i, job in enumerate(_structured):
        _, st = gen_one(*job); written += st == "written"; skipped += st == "skipped"
        if (i + 1) % 250 == 0:
            print(f"  [{i+1:,}/{len(_structured):,}] written={written:,} skipped={skipped:,}")
    for s in _noise:
        _, st = gen_noise(s); written += st == "written"; skipped += st == "skipped"
    print(f"done. written={written:,} skipped={skipped:,}  -> {GEN_OUTDIR}")

model x shape combos : 26
structured fields    : 12,480
pure-noise fields    : 6
TOTAL                : 12,486   (~44 GB est. at (1000, 1000) float32)
output dir           : synthetic_benchmark/grf_dataset/
  [250/12,480] written=0 skipped=250
  [500/12,480] written=0 skipped=500
  [750/12,480] written=0 skipped=750
  [1,000/12,480] written=0 skipped=1,000
  [1,250/12,480] written=0 skipped=1,250
  [1,500/12,480] written=0 skipped=1,500
  [1,750/12,480] written=0 skipped=1,750
  [2,000/12,480] written=0 skipped=2,000
  [2,250/12,480] written=0 skipped=2,250
  [2,500/12,480] written=0 skipped=2,500
  [2,750/12,480] written=0 skipped=2,750
  [3,000/12,480] written=0 skipped=3,000
  [3,250/12,480] written=0 skipped=3,250
  [3,500/12,480] written=0 skipped=3,500
  [3,750/12,480] written=0 skipped=3,750
  [4,000/12,480] written=0 skipped=4,000
  [4,250/12,480] written=0 skipped=4,250
  [4,500/12,480] written=0 skipped=4,500
  [4,750/12,480] written=0 skipped=4,750
  [5,000/12,480] written=0

### Nested (multi-component) models

Each field is a **sum of independent component GRFs** + nugget. Templates vary the **range-separation ratio** (a ramp ~1.5x -> 50x, to map where nested scales become separable) and the **sill split**; sweep is reduced (sill in {0.1,1}, nugget in {0,0.2,0.5}). Set `GENERATE_NESTED = True` to write.

In [4]:
# ---------- nested (multi-component) model templates ----------
NESTED_SILLS        = [0.1, 1.0]
NESTED_NUGGET_FRACS = [0.0, 0.2, 0.5]
NESTED_SPLITS = {                                   # sill weights (fractions of structured sill)
    2: [(0.4, 0.6), (0.9, 0.1), (0.2, 0.8)],
    3: [(1/3, 1/3, 1/3), (0.6, 0.3, 0.1), (0.2, 0.3, 0.5)],
}

# template name -> ordered components [(model, shape_val, range), ...]; weights come from NESTED_SPLITS
NESTED_TEMPLATES = {
    # separation ramp (2x Spherical, short=20 m): ratio 1.5 .. 50x -> where do the scales separate?
    **{f"Sph20_Sph{int(round(20*r))}": [("Spherical", None, 20), ("Spherical", None, int(round(20*r)))]
       for r in [1.5, 2, 3, 5, 10, 25, 50]},
    # explicit examples
    "Sph10_Sph100":         [("Spherical", None, 10),  ("Spherical", None, 100)],
    "Sph100_Sph1000":       [("Spherical", None, 100), ("Spherical", None, 1000)],
    "Sph10_Sph1000":        [("Spherical", None, 10),  ("Spherical", None, 1000)],
    # 3-component (decreasing separation: 10x/10x, 5x/5x, 3x/3.3x)
    "Sph10_Sph100_Sph1000": [("Spherical", None, 10), ("Spherical", None, 100), ("Spherical", None, 1000)],
    "Sph10_Sph50_Sph250":   [("Spherical", None, 10), ("Spherical", None, 50),  ("Spherical", None, 250)],
    "Sph20_Sph60_Sph200":   [("Spherical", None, 20), ("Spherical", None, 60),  ("Spherical", None, 200)],
    # mixed model types
    "Mat50_Sph500":         [("Matern", 1.5, 50),  ("Spherical", None, 500)],
    "Exp20_Sph200":         [("Exponential", None, 20), ("Spherical", None, 200)],
    "Sph30_Gau300":         [("Spherical", None, 30), ("Gaussian", None, 300)],
    "Sta50_Sph500":         [("Stable", 1.5, 50), ("Spherical", None, 500)],
    "Mat10_Mat100":         [("Matern", 0.5, 10), ("Matern", 2.5, 100)],
}

def gen_components(weighted_comps, total_sill, nug_frac, fname):
    """Sum of independent component GRFs + nugget. weighted_comps: [(model, shape_val, range, weight)]."""
    out = os.path.join(GEN_OUTDIR, fname)
    if os.path.exists(out):
        return out, "skipped"
    structured, nugget = (1 - nug_frac) * total_sill, nug_frac * total_sill
    sd = _seed_for(fname)
    field = np.zeros(GEN_SHAPE)
    meta = []
    for ci, (mname, sv, rng, w) in enumerate(weighted_comps):
        cls = getattr(gs, mname); param = GEN_MODELS[mname]["param"]
        kw = {param: sv} if (param and sv is not None) else {}
        cvar = w * structured
        ls = gen_len_scale(cls, rng, **kw)
        field = field + grf_circulant(cls(dim=2, var=cvar, len_scale=ls, **kw),
                                      GEN_SHAPE, GEN_RES, seed=sd + ci, pad=gen_pad(rng))
        meta.append({"model": mname, "param": param, "value": sv, "range": rng,
                     "weight": w, "partial_sill": cvar, "len_scale": ls})
    if nugget > 0:
        field = field + np.sqrt(nugget) * np.random.default_rng(sd + 100).normal(size=GEN_SHAPE)
    ny, nx = GEN_SHAPE
    _write(out, field, dict(
        variogram_model=f"nested{len(weighted_comps)}", n_components=len(weighted_comps),
        components=json.dumps(meta), sill=total_sill, partial_sill=structured, nugget=nugget,
        nugget_frac=nug_frac, range="", range_percentile=GEN_PERCENTILE,
        generator="circulant_embedding_nested", seed=sd, shape=f"{ny}x{nx}"))
    return out, "written"

def build_nested_jobs():
    ny, nx = GEN_SHAPE; jobs = []
    for key, comps in NESTED_TEMPLATES.items():
        k = len(comps)
        for split in NESTED_SPLITS[k]:
            wcomps = [(m, sv, rng, w) for (m, sv, rng), w in zip(comps, split)]
            ws = "-".join(str(int(round(w * 100))) for w in split)
            for sill in NESTED_SILLS:
                for f in NESTED_NUGGET_FRACS:
                    fname = f"nest{k}_{key}_w{ws}_s{_fmt(sill)}_n{int(round(f*100))}_{ny}x{nx}.tif"
                    jobs.append((wcomps, sill, f, fname))
    return jobs

In [5]:
_nested = build_nested_jobs()
print(f"nested templates : {len(NESTED_TEMPLATES)}  ({sum(len(c)==2 for c in NESTED_TEMPLATES.values())} two-comp, {sum(len(c)==3 for c in NESTED_TEMPLATES.values())} three-comp)")
print(f"nested fields    : {len(_nested):,}   (~{len(_nested)*3.5/1000:.1f} GB est. at {GEN_SHAPE} {GEN_DTYPE})")
print(f"output dir       : {GEN_OUTDIR}")

GENERATE_NESTED = True    # <-- set True to write the nested dataset (resumable: skips existing files)

if GENERATE_NESTED:
    written = skipped = 0
    for i, (wcomps, sill, f, fname) in enumerate(_nested):
        _, st = gen_components(wcomps, sill, f, fname)
        written += st == "written"; skipped += st == "skipped"
        if (i + 1) % 50 == 0:
            print(f"  [{i+1:,}/{len(_nested):,}] written={written:,} skipped={skipped:,}")
    print(f"done. written={written:,} skipped={skipped:,}  -> {GEN_OUTDIR}")

nested templates : 18  (15 two-comp, 3 three-comp)
nested fields    : 324   (~1.1 GB est. at (1000, 1000) float32)
output dir       : synthetic_benchmark/grf_dataset/
  [50/324] written=0 skipped=50
  [100/324] written=0 skipped=100
  [150/324] written=0 skipped=150
  [200/324] written=0 skipped=200
  [250/324] written=0 skipped=250
  [300/324] written=0 skipped=300
done. written=0 skipped=324  -> synthetic_benchmark/grf_dataset/


## Additional scenarios (systematic error, trend, anisotropy)

All layered on a shared **nested-spherical base** (`BASE_SPHERICAL` x `BASE_NUGGETS`). Each section is gated by its own `GENERATE_*` flag and writes into `GEN_OUTDIR` (so the dataset catalog picks them up).

In [6]:
# ===== shared base + overlays for the extra scenarios (systematic error / trend / anisotropy) =====
# Base = nested-spherical field (equal-weight components) + optional nugget; overlays are added on top.
BASE_SPHERICAL = {
    **{f"Sph{r}": [r] for r in [5, 10, 50, 100, 500, 1000, 2000]},   # single-component
    "Sph20_Sph200":  [20, 200],   # 2-comp, 10x separation
    "Sph20_Sph60":   [20, 60],    # 2-comp, 3x
    "Sph50_Sph500":  [50, 500],   # 2-comp, 10x
    "Sph10_Sph100_Sph1000": [10, 100, 1000],   # 3-comp
    "Sph20_Sph60_Sph200":   [20, 60, 200],     # 3-comp
}
BASE_NUGGETS = [0.0, 0.2]
SCEN_SILL = 1.0
SCEN_SHAPE, SCEN_RES = GEN_SHAPE, GEN_RES

def base_seed(name, nug):
    return _seed_for(f"base_{name}_n{int(round(nug*100))}")

def base_array(ranges, nug_frac, S=SCEN_SILL, seed=0):
    """Nested-spherical base field: equal-weight spherical components + nugget."""
    structured = (1 - nug_frac) * S; w = structured / len(ranges)
    field = np.zeros(SCEN_SHAPE)
    for ci, rng in enumerate(ranges):
        m = gs.Spherical(dim=2, var=w, len_scale=rng)      # spherical: len_scale == range
        field = field + grf_circulant(m, SCEN_SHAPE, SCEN_RES, seed=seed + ci, pad=gen_pad(rng))
    if nug_frac > 0:
        field = field + np.sqrt(nug_frac * S) * np.random.default_rng(seed + 50).normal(size=SCEN_SHAPE)
    return field

def base_tag(name, nug_frac):
    ranges = BASE_SPHERICAL[name]; w = (1 - nug_frac) * SCEN_SILL / len(ranges)
    comps = [{"model": "Spherical", "range": r, "partial_sill": w, "nu": None, "alpha": None} for r in ranges]
    return dict(base=name, n_components=len(ranges), components=json.dumps(comps),
                sill=SCEN_SILL, partial_sill=(1 - nug_frac) * SCEN_SILL,
                nugget=nug_frac * SCEN_SILL, nugget_frac=nug_frac, shape=f"{SCEN_SHAPE[0]}x{SCEN_SHAPE[1]}")

def _xy_grid():
    ny, nx = SCEN_SHAPE
    return np.meshgrid(np.arange(nx) * SCEN_RES, np.arange(ny) * SCEN_RES)   # X (cols), Y (rows)

# ---- trend surfaces ----
def planar_trend(slope, direction_deg):
    X, Y = _xy_grid(); th = np.radians(direction_deg)
    return slope * (np.cos(th) * X + np.sin(th) * Y)

POLY_SURFACES = {
    "bowl":   lambda Xn, Yn: Xn**2 + Yn**2,
    "saddle": lambda Xn, Yn: Xn**2 - Yn**2,
    "tilt2":  lambda Xn, Yn: Xn**2 + 0.5 * Xn * Yn,
    "ridge3": lambda Xn, Yn: Xn**3 - 3 * Xn * Yn**2,     # cubic
    "dome3":  lambda Xn, Yn: -(Xn**2 + Yn**2) + 0.5 * Xn**3,
}
def poly_trend(name, amp):
    X, Y = _xy_grid(); ny, nx = SCEN_SHAPE
    Xn = X / ((nx - 1) * SCEN_RES) * 2 - 1; Yn = Y / ((ny - 1) * SCEN_RES) * 2 - 1
    s = POLY_SURFACES[name](Xn, Yn)
    return amp * s / np.max(np.abs(s))                    # unit-peak * amplitude

_BAND_ANGLE = {"vertical": 90, "horizontal": 0, "diag45": 45, "diag135": 135}
def band_trend(amp, wavelength, orient):
    X, Y = _xy_grid(); th = np.radians(_BAND_ANGLE[orient])
    return amp * np.sin(2 * np.pi * (np.cos(th) * X + np.sin(th) * Y) / wavelength)

# ---- anisotropic component (circulant embedding on the anisotropic covariance) ----
def _sph_corr(d):
    d = np.asarray(d, float); return np.where(d < 1.0, 1 - 1.5 * d + 0.5 * d**3, 0.0)
def aniso_component(amp, major, ratio, angle_deg, seed=0, pad=2):
    ny, nx = SCEN_SHAPE; My, Mx = pad * ny, pad * nx
    ky = np.concatenate([np.arange(0, My // 2 + 1), np.arange(My // 2 + 1 - My, 0)])
    kx = np.concatenate([np.arange(0, Mx // 2 + 1), np.arange(Mx // 2 + 1 - Mx, 0)])
    Y, X = np.meshgrid(ky * SCEN_RES, kx * SCEN_RES, indexing="ij")
    th = np.radians(angle_deg); c, s = np.cos(th), np.sin(th)
    xr = c * X + s * Y; yr = -s * X + c * Y                # rotate lag into principal frame
    d = np.sqrt((xr / major)**2 + (yr / (major / ratio))**2)
    lam = np.fft.fft2(amp * _sph_corr(d)).real; lam[lam < 0] = 0.0
    r = np.random.default_rng(seed); xi = r.normal(size=(My, Mx)) + 1j * r.normal(size=(My, Mx))
    return (np.fft.fft2(xi * np.sqrt(lam)) / np.sqrt(My * Mx)).real[:ny, :nx]

In [7]:
### A. Systematic error (constant bias)
# base (or pure noise) + a constant offset b. A constant offset leaves the variogram unchanged
# (Matheron differences cancel it) -> tests median-based bias estimation and bias-vs-random areal uncertainty.
SYSERR_BIAS = [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 30]

def gen_syserr():
    ny, nx = SCEN_SHAPE; written = skipped = 0
    for name, ranges in BASE_SPHERICAL.items():
        for nug in BASE_NUGGETS:
            base = None
            for b in SYSERR_BIAS:
                fn = f"syserr_{name}_n{int(round(nug*100))}_b{_fmt(b)}_{ny}x{nx}.tif"
                out = os.path.join(GEN_OUTDIR, fn)
                if os.path.exists(out): skipped += 1; continue
                if base is None: base = base_array(ranges, nug, seed=base_seed(name, nug))
                _write(out, base + b, {**base_tag(name, nug), "kind": "syserr", "syserr": b,
                                       "generator": "base+bias", "seed": base_seed(name, nug)})
                written += 1
    for b in SYSERR_BIAS:                                   # random uncorrelated field + bias
        fn = f"syserr_noise_b{_fmt(b)}_{ny}x{nx}.tif"; out = os.path.join(GEN_OUTDIR, fn)
        if os.path.exists(out): skipped += 1; continue
        noise = np.sqrt(SCEN_SILL) * np.random.default_rng(_seed_for(fn)).normal(size=SCEN_SHAPE)
        _write(out, noise + b, {"kind": "syserr", "base": "noise", "n_components": 0, "components": "[]",
                                "sill": SCEN_SILL, "partial_sill": 0.0, "nugget": SCEN_SILL, "nugget_frac": 1.0,
                                "syserr": b, "generator": "noise+bias", "shape": f"{ny}x{nx}"})
        written += 1
    print(f"syserr: written={written} skipped={skipped}")

_n = len(BASE_SPHERICAL) * len(BASE_NUGGETS) * len(SYSERR_BIAS) + len(SYSERR_BIAS)
print(f"systematic-error fields: {_n:,}   (~{_n*4/1000:.1f} GB)")
GENERATE_SYSERR = False    # <-- set True to write
if GENERATE_SYSERR: gen_syserr()

systematic-error fields: 225   (~0.9 GB)


In [8]:
### B. Trend surfaces (planar / polynomial / alternating bands) added to the base
PLANAR_SLOPES = [0.001, 0.01, 0.1, 0.5]
PLANAR_DIRS   = [0, 45, 90, 135]
POLY_AMPS     = [0.5, 2.0]
BAND_AMPS     = [0.005, 0.01, 0.05, 0.1, 0.5, 1, 2]
BAND_LAMBDAS  = [25, 50, 200, 500, 1000, 2000]
BAND_ORIENTS  = ["vertical", "horizontal", "diag45", "diag135"]

def _trend_jobs_for(name, nug):
    """(filename, trend_surface_callable, tag_dict) for one base."""
    ny, nx = SCEN_SHAPE; p = f"{name}_n{int(round(nug*100))}"; jobs = []
    for sl in PLANAR_SLOPES:
        for d in PLANAR_DIRS:
            jobs.append((f"trend_{p}_planar_sl{_fmt(sl)}_d{d}_{ny}x{nx}.tif",
                         lambda sl=sl, d=d: planar_trend(sl, d),
                         {"trend_type": "planar", "slope": sl, "direction": d}))
    for pn in POLY_SURFACES:
        for a in POLY_AMPS:
            jobs.append((f"trend_{p}_poly-{pn}_a{_fmt(a)}_{ny}x{nx}.tif",
                         lambda pn=pn, a=a: poly_trend(pn, a),
                         {"trend_type": "poly", "poly": pn, "amp": a}))
    for a in BAND_AMPS:
        for lam in BAND_LAMBDAS:
            for o in BAND_ORIENTS:
                jobs.append((f"trend_{p}_bands_a{_fmt(a)}_l{lam}_o-{o}_{ny}x{nx}.tif",
                             lambda a=a, lam=lam, o=o: band_trend(a, lam, o),
                             {"trend_type": "bands", "amp": a, "wavelength": lam, "orient": o}))
    return jobs

def gen_trend():
    written = skipped = 0
    for name, ranges in BASE_SPHERICAL.items():
        for nug in BASE_NUGGETS:
            jobs = _trend_jobs_for(name, nug); base = None
            for fn, surf, ttag in jobs:
                out = os.path.join(GEN_OUTDIR, fn)
                if os.path.exists(out): skipped += 1; continue
                if base is None: base = base_array(ranges, nug, seed=base_seed(name, nug))
                _write(out, base + surf(), {**base_tag(name, nug), "kind": "trend",
                                            **ttag, "generator": "base+trend", "seed": base_seed(name, nug)})
                written += 1
            if written and written % 200 < len(jobs): print(f"  trend {name} n{int(nug*100)}: {written} written")
    print(f"trend: written={written} skipped={skipped}")

_per = 16 + len(POLY_SURFACES)*len(POLY_AMPS) + len(BAND_AMPS)*len(BAND_LAMBDAS)*len(BAND_ORIENTS)
_n = _per * len(BASE_SPHERICAL) * len(BASE_NUGGETS)
print(f"trend fields: {_per}/base x {len(BASE_SPHERICAL)*len(BASE_NUGGETS)} bases = {_n:,}   (~{_n*4/1000:.1f} GB)")
GENERATE_TREND = False    # <-- set True to write
if GENERATE_TREND: gen_trend()

trend fields: 194/base x 24 bases = 4,656   (~18.6 GB)


In [9]:
### C. Anisotropic component added to the base
ANI_AMPS   = [0.1, 1]                                    # variance of the anisotropic component
ANI_DIRS   = [0, 45, 90, 135]
ANI_RATIOS = [2, 4, 8, 16]                                # major:minor range
ANI_MAJORS = [10, 100, 500]                               # major-axis range

def gen_aniso():
    ny, nx = SCEN_SHAPE; written = skipped = 0
    for name, ranges in BASE_SPHERICAL.items():
        for nug in BASE_NUGGETS:
            base = None
            for amp in ANI_AMPS:
                for d in ANI_DIRS:
                    for rat in ANI_RATIOS:
                        for maj in ANI_MAJORS:
                            fn = f"aniso_{name}_n{int(round(nug*100))}_amp{_fmt(amp)}_d{d}_r{rat}_maj{maj}_{ny}x{nx}.tif"
                            out = os.path.join(GEN_OUTDIR, fn)
                            if os.path.exists(out): skipped += 1; continue
                            if base is None: base = base_array(ranges, nug, seed=base_seed(name, nug))
                            an = aniso_component(amp, maj, rat, d, seed=_seed_for(fn), pad=gen_pad(maj))
                            _write(out, base + an, {**base_tag(name, nug), "kind": "aniso",
                                    "ani_amp": amp, "ani_dir": d, "ani_ratio": rat, "ani_major": maj,
                                    "generator": "base+aniso", "seed": _seed_for(fn)})
                            written += 1
            if written: print(f"  aniso {name} n{int(nug*100)}: cumulative {written}")
    print(f"aniso: written={written} skipped={skipped}")

_per = len(ANI_AMPS)*len(ANI_DIRS)*len(ANI_RATIOS)*len(ANI_MAJORS)
_n = _per * len(BASE_SPHERICAL) * len(BASE_NUGGETS)
print(f"aniso fields: {_per}/base x {len(BASE_SPHERICAL)*len(BASE_NUGGETS)} bases = {_n:,}   (~{_n*4/1000:.0f} GB)")
GENERATE_ANISO = False    # <-- set True to write
if GENERATE_ANISO: gen_aniso()

aniso fields: 96/base x 24 bases = 2,304   (~9 GB)


In [11]:
### D. All perturbations combined  (base + systematic error + trend + anisotropy)
# Trimmed factorial: 24 bases x 2 biases x 4 trends x 4 aniso configs = 768 fields.
COMBO_SYSERR = [0.1, 5.0]
COMBO_TRENDS = [
    ("planar", dict(slope=0.01, direction=0)),
    ("planar", dict(slope=0.1,  direction=45)),
    ("poly",   dict(poly="bowl", amp=1.0)),
    ("bands",  dict(amp=0.5, wavelength=200, orient="vertical")),
]
COMBO_ANISO = [
    dict(amp=1.0, major=100, ratio=4,  dir=45),
    dict(amp=0.5, major=50,  ratio=8,  dir=90),
    dict(amp=1.0, major=500, ratio=2,  dir=0),
    dict(amp=0.1, major=100, ratio=16, dir=135),
]

def _trend_surface(ttype, p):
    if ttype == "planar": return planar_trend(p["slope"], p["direction"])
    if ttype == "poly":   return poly_trend(p["poly"], p["amp"])
    if ttype == "bands":  return band_trend(p["amp"], p["wavelength"], p["orient"])

def _trend_token(ttype, p):
    if ttype == "planar": return f"planar-sl{_fmt(p['slope'])}-d{p['direction']}"
    if ttype == "poly":   return f"poly-{p['poly']}-a{_fmt(p['amp'])}"
    if ttype == "bands":  return f"bands-a{_fmt(p['amp'])}-l{p['wavelength']}-o-{p['orient']}"

def gen_combo():
    ny, nx = SCEN_SHAPE; written = skipped = 0
    for name, ranges in BASE_SPHERICAL.items():
        for nug in BASE_NUGGETS:
            base = None
            for b in COMBO_SYSERR:
                for ttype, tp in COMBO_TRENDS:
                    for ap in COMBO_ANISO:
                        tok = _trend_token(ttype, tp)
                        fn = (f"combo_{name}_n{int(round(nug*100))}_b{_fmt(b)}_{tok}"
                              f"_ani-a{_fmt(ap['amp'])}-d{ap['dir']}-r{ap['ratio']}-m{ap['major']}_{ny}x{nx}.tif")
                        out = os.path.join(GEN_OUTDIR, fn)
                        if os.path.exists(out): skipped += 1; continue
                        if base is None: base = base_array(ranges, nug, seed=base_seed(name, nug))
                        an = aniso_component(ap["amp"], ap["major"], ap["ratio"], ap["dir"],
                                             seed=_seed_for(fn), pad=gen_pad(ap["major"]))
                        field = base + b + _trend_surface(ttype, tp) + an
                        _write(out, field, {**base_tag(name, nug), "kind": "combo", "syserr": b,
                               "trend_type": ttype, **{f"trend_{k}": v for k, v in tp.items()},
                               "ani_amp": ap["amp"], "ani_dir": ap["dir"], "ani_ratio": ap["ratio"], "ani_major": ap["major"],
                               "generator": "base+bias+trend+aniso", "seed": _seed_for(fn)})
                        written += 1
            if written: print(f"  combo {name} n{int(nug*100)}: cumulative {written}")
    print(f"combo: written={written} skipped={skipped}")

_n = len(BASE_SPHERICAL) * len(BASE_NUGGETS) * len(COMBO_SYSERR) * len(COMBO_TRENDS) * len(COMBO_ANISO)
print(f"combined-perturbation fields: {_n:,}   (~{_n*4/1000:.1f} GB)")
GENERATE_COMBO = True    # <-- set True to write
if GENERATE_COMBO: gen_combo()

combined-perturbation fields: 768   (~3.1 GB)
  combo Sph5 n0: cumulative 32
  combo Sph5 n20: cumulative 64
  combo Sph10 n0: cumulative 96
  combo Sph10 n20: cumulative 128
  combo Sph50 n0: cumulative 160
  combo Sph50 n20: cumulative 192
  combo Sph100 n0: cumulative 224
  combo Sph100 n20: cumulative 256
  combo Sph500 n0: cumulative 288
  combo Sph500 n20: cumulative 320
  combo Sph1000 n0: cumulative 352
  combo Sph1000 n20: cumulative 384
  combo Sph2000 n0: cumulative 416
  combo Sph2000 n20: cumulative 448
  combo Sph20_Sph200 n0: cumulative 480
  combo Sph20_Sph200 n20: cumulative 512
  combo Sph20_Sph60 n0: cumulative 544
  combo Sph20_Sph60 n20: cumulative 576
  combo Sph50_Sph500 n0: cumulative 608
  combo Sph50_Sph500 n20: cumulative 640
  combo Sph10_Sph100_Sph1000 n0: cumulative 672
  combo Sph10_Sph100_Sph1000 n20: cumulative 704
  combo Sph20_Sph60_Sph200 n0: cumulative 736
  combo Sph20_Sph60_Sph200 n20: cumulative 768
combo: written=768 skipped=0


## Parameters and helpers

In [ ]:
import json, numpy as np, pandas as pd, math, rasterio, shapely, geopandas as gpd
import gstools as gs
from skgstat import Variogram
from topochange import RasterDataHandler, SingleVariogram, MODEL_REGISTRY

# ---- experiment parameters ----
model_name = "Spherical"
ranges = [5, 10, 25, 50, 100, 250, 500, 1000, 1500, 2000]
sills = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10]
percentile, nugget = 0.95, 0.0
shape, resolution, mean = (500, 500), 1.0, 0.0
origin, crs, dtype = (500000.0, 5000000.0), "EPSG:32633", "float32"
outpath = "synthetic_benchmark/figures/out_tif/"
test_path = outpath + "Spherical_50_0.1_500_500"

# ---- raster helpers ----
def raster_xy(path):
    with rasterio.open(path) as src:
        t = src.transform
        x = t.c + t.a * (np.arange(src.width) + 0.5)
        y = t.f + t.e * (np.arange(src.height) + 0.5)
        data = src.read(1)
    return x, y, data

def max_lag_for(path, frac=1/3):
    with rasterio.open(path) as src:
        b = src.bounds
    return frac * np.hypot(b.right - b.left, b.top - b.bottom)

# ---- gstools fit helpers ----
_FINITE_RANGE = {"Spherical", "Circular", "Cubic", "HyperSpherical", "SuperSpherical"}
def effective_range(fm, name, percentile=0.95):
    if name in _FINITE_RANGE:
        return float(fm.len_scale)
    return float(fm.percentile_scale(percentile))
GSTOOLS_MODELS = {"Gaussian": gs.Gaussian, "Exponential": gs.Exponential,
                  "Matern": gs.Matern, "Stable": gs.Stable, "Spherical": gs.Spherical}

# ---- scikit-gstat fit helpers ----
def load_tif_variogram(path, sampling_size=2000, n_lags=30, model="spherical", maxlag=None, seed=42):
    with rasterio.open(path) as src:
        t = src.transform
        x = t.c + t.a * (np.arange(src.width) + 0.5)
        y = t.f + t.e * (np.arange(src.height) + 0.5)
        field = src.read(1).astype(float); nodata = src.nodata
    xx, yy = np.meshgrid(x, y, indexing="xy")
    coords = np.column_stack([xx.ravel(), yy.ravel()]); vals = field.ravel()
    m = np.isfinite(vals)
    if nodata is not None: m &= vals != nodata
    coords, vals = coords[m], vals[m]
    if sampling_size and sampling_size < len(vals):
        idx = np.random.default_rng(seed).choice(len(vals), sampling_size, replace=False)
        coords, vals = coords[idx], vals[idx]
    return Variogram(coords, vals, n_lags=n_lags, model=model, maxlag=maxlag)

skg_models = ["spherical", "exponential", "gaussian", "cubic", "stable", "matern"]
def parse_skg(V, name):
    p = list(V.parameters); er, ps = p[0], p[1]
    if name in ("matern", "stable"): sh, ng = p[2], p[3]
    else: sh, ng = np.nan, p[2]
    return er, ps, ng, (sh if name == "matern" else np.nan), (sh if name == "stable" else np.nan)
def pseudo_r2(V):
    e = np.asarray(V.experimental, float); mo = np.asarray(V.fitted_model(V.bins), float)
    k = np.isfinite(e) & np.isfinite(mo)
    sr = np.sum((e[k] - mo[k])**2); st = np.sum((e[k] - e[k].mean())**2)
    return 1 - sr / st if st > 0 else np.nan

# ---- common schema + truth ----
COMMON_COLS = ["method", "best_model", "pseudo_r2", "var", "sill", "nugget", "range", "nu", "alpha"]
def truth_from_tags(path):
    """Ground-truth params from the GeoTIFF tags (single + nested models)."""
    with rasterio.open(path) as src: t = src.tags()
    nug = float(t.get("nugget", 0.0) or 0.0)
    if "partial_sill" in t: var, total = float(t["partial_sill"]), float(t["sill"])
    else: var = float(t["sill"]); total = var + nug
    base = {"method": "truth", "best_model": t.get("variogram_model"), "pseudo_r2": 1.0,
            "var": var, "sill": total, "nugget": nug}
    if t.get("components"):                       # nested (multi-component) field
        return {**base, "range": np.nan, "nu": np.nan, "alpha": np.nan,
                "components": json.loads(t["components"])}
    def _num(k):
        v = t.get(k, "")
        try: return float(v)
        except (TypeError, ValueError): return np.nan
    return {**base, "range": float(t["range"]), "nu": _num("nu"), "alpha": _num("alpha")}


## Dataset selection

Scan the generated dataset into a `catalog`, then pick a `SELECTION` (a slice of it). `SELECTION` drives both the fitting and areal sections, so you can run any subset (single or nested).

In [12]:
import os, glob, re
import numpy as np, pandas as pd

DATA_DIR = "synthetic_benchmark/grf_dataset/"     # generated dataset (matches GEN_OUTDIR)

def _unfmt(tok): return float(tok.replace("-", "."))   # '0-001'->0.001, '1-5'->1.5

_RE_NUGGET = re.compile(r"^Nugget_s(?P<sill>[0-9-]+)_n100_(?P<H>\d+)x(?P<W>\d+)\.tif$")
_RE_NESTED = re.compile(r"^nest(?P<k>\d)_(?P<tmpl>.+?)_w(?P<split>[0-9-]+)_s(?P<sill>[0-9-]+)_n(?P<nug>\d+)_(?P<H>\d+)x(?P<W>\d+)\.tif$")
_RE_SINGLE = re.compile(r"^(?P<model>[A-Za-z]+)_r(?P<rng>[0-9-]+)_s(?P<sill>[0-9-]+)_n(?P<nug>\d+)(?:_nu(?P<nu>[0-9-]+)|_a(?P<alpha>[0-9-]+))?_(?P<H>\d+)x(?P<W>\d+)\.tif$")

def parse_name(fn):
    m = re.match(r"^(?P<kind>syserr|trend|aniso|combo)_", fn)
    if m:
        kind = m["kind"]
        d = dict(kind=kind, model=kind, n_components=np.nan, range=np.nan, sill=np.nan,
                 nugget_frac=np.nan, nu=np.nan, alpha=np.nan, separation_ratio=np.nan)
        b = re.search(r"_n(\d+)_", fn)
        if b: d["nugget_frac"] = int(b.group(1)) / 100
        for key, pat in (("syserr", r"_b([0-9-]+)_"), ("slope", r"_sl([0-9-]+)_"),
                         ("ani_ratio", r"_r(\d+)_"), ("ani_major", r"_maj(\d+)_")):
            mm = re.search(pat, fn)
            if mm: d[key] = _unfmt(mm.group(1)) if "-" in mm.group(1) or key in ("syserr","slope") else float(mm.group(1))
        return d
    m = _RE_NUGGET.match(fn)
    if m:
        return dict(kind="nugget", model="Nugget", n_components=0, range=np.nan,
                    sill=_unfmt(m["sill"]), nugget_frac=1.0, nu=np.nan, alpha=np.nan, separation_ratio=np.nan)
    m = _RE_NESTED.match(fn)
    if m:
        rng = [int(x) for x in re.findall(r"\d+", m["tmpl"])]
        return dict(kind="nested", model=f"nested{m['k']}", n_components=int(m["k"]), range=np.nan,
                    sill=_unfmt(m["sill"]), nugget_frac=int(m["nug"]) / 100, nu=np.nan, alpha=np.nan,
                    separation_ratio=round(max(rng) / min(rng), 2) if rng else np.nan)
    m = _RE_SINGLE.match(fn)
    if m:
        return dict(kind="single", model=m["model"], n_components=1, range=_unfmt(m["rng"]),
                    sill=_unfmt(m["sill"]), nugget_frac=int(m["nug"]) / 100,
                    nu=_unfmt(m["nu"]) if m["nu"] else np.nan,
                    alpha=_unfmt(m["alpha"]) if m["alpha"] else np.nan, separation_ratio=np.nan)
    return None

def build_catalog(data_dir=DATA_DIR):
    rows = []
    for p in sorted(glob.glob(os.path.join(data_dir, "*.tif"))):
        meta = parse_name(os.path.basename(p))
        if meta is not None:
            rows.append({"file": os.path.basename(p), "path": p, **meta})
    return pd.DataFrame(rows)

catalog = build_catalog()
print(f"{len(catalog)} rasters in {DATA_DIR}")
if len(catalog):
    print(catalog.groupby("kind").size().to_string())
    print("models:", sorted(catalog.model.unique()))
catalog.head()

20763 rasters in synthetic_benchmark/grf_dataset/
kind
aniso      2304
combo       768
nested      324
nugget        6
single    12480
syserr      225
trend      4656
models: ['Circular', 'Cubic', 'Exponential', 'Gaussian', 'HyperSpherical', 'Integral', 'JBessel', 'Matern', 'Nugget', 'Rational', 'Spherical', 'Stable', 'SuperSpherical', 'aniso', 'combo', 'nested2', 'nested3', 'syserr', 'trend']


,file,path,kind,model,n_components,range,sill,nugget_frac,nu,alpha,separation_ratio,ani_ratio,ani_major,syserr,slope
0,Circular_r1000_s0-001_n0_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Circular_r1000_s0-001_n10_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Circular_r1000_s0-001_n20_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Circular_r1000_s0-001_n30_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Circular_r1000_s0-001_n40_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ===== choose the subsection to run through BOTH sections =====
# Edit this filter; SELECTION (a slice of `catalog`) drives the fitting + areal runs.
SELECTION = catalog[
    (catalog.kind == "single")
    & (catalog.model == "Spherical")
    & (catalog.nugget_frac == 0.0)
    & (catalog.sill == 1.0)
].reset_index(drop=True)

# SELECTION = catalog
# more examples (adapt):
#   nested, well separated:  catalog[(catalog.kind=="nested") & (catalog.separation_ratio >= 10)]
#   a nugget sweep:          catalog[(catalog.kind=="single") & (catalog.model=="Matern") & (catalog.range==50) & (catalog.sill==1.0)]
#   several families:        catalog[(catalog.kind=="single") & (catalog.model.isin(["Spherical","Exponential","Gaussian"]))]
#   everything:              catalog

DEV_PATH = SELECTION["path"].iloc[0] if len(SELECTION) else None
print(f"SELECTION: {len(SELECTION)} files" + (f"   DEV_PATH={os.path.basename(DEV_PATH)}" if DEV_PATH else ""))
SELECTION.head(10)

SELECTION: 20763 files   DEV_PATH=Circular_r1000_s0-001_n0_1000x1000.tif


,file,path,kind,model,n_components,range,sill,nugget_frac,nu,alpha,separation_ratio,ani_ratio,ani_major,syserr,slope
0,Circular_r1000_s0-001_n0_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Circular_r1000_s0-001_n10_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Circular_r1000_s0-001_n20_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Circular_r1000_s0-001_n30_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Circular_r1000_s0-001_n40_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Circular_r1000_s0-001_n50_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Circular_r1000_s0-001_n60_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Circular_r1000_s0-001_n70_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Circular_r1000_s0-001_n80_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Circular_r1000_s0-001_n90_1000x1000.tif,synthetic_benchmark/grf_dataset/Circular_r1000...,single,Circular,1.0,1000.0,0.001,0.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Variogram fitting comparison

Fit every raster in `SELECTION` with each library and score recovered-vs-true parameters. Single-model and nested fields are handled uniformly (single = 1 component).

### Per-library fit functions

In [14]:
from topochange import RasterDataHandler, SingleVariogram, MODEL_REGISTRY

def fit_topochange(path, resolution=1.0, max_samples=3000,
                   bin_width=None, seed=42, model_types=None):
    """Fit a variogram with topochange; return the common schema."""
    rdh = RasterDataHandler(path, unit="m", resolution=resolution)
    rdh.load_raster()                              # sets the rioxarray obj (needed for extent)
    sv = SingleVariogram(rdh)

    sv.compute_empirical_variogram(
        area_side=1.0, samples_per_area=1.0, max_samples=max_samples,
        bin_width=bin_width or (2 * resolution),
        max_lag_multiplier=1 / 3, seed=seed,
        estimator="matheron", return_sample=True,
    )
    sv.fit_model(
        model_types=model_types or ["spherical", "exponential", "matern"],
        include_nugget=True, criterion="aicc", seed=seed,
    )

    bm = sv.best_model
    model = bm["model"]
    model.set_params(bm["params"])
    pdict = dict(zip(model.param_names, np.asarray(bm["params"], float)))

    comp = model.component_names[0] if model.component_names else "nugget"
    raw_range = next((v for k, v in pdict.items() if k.endswith("range")), np.nan)
    # topochange stores a RAW range; effective range = raw * practical_range_factor
    prf = MODEL_REGISTRY.get_model(comp).practical_range_factor if comp != "nugget" else np.nan
    nugget = model.get_nugget()
    sill = model.get_total_sill()
    pcov = bm.get("param_cov")
    pstd = dict(zip(model.param_names, np.sqrt(np.abs(np.diag(pcov))))) if pcov is not None else {}

    g = sv.variogram                                # unweighted pseudo-R^2, comparable across libs
    pred = model(sv.lags)                           # composite model is callable
    ss_res = float(np.sum((g - pred) ** 2))
    ss_tot = float(np.sum((g - g.mean()) ** 2))
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return {
        "method": "topochange",
        "best_model": comp.capitalize(),
        "pseudo_r2": r2,
        "var": sill - nugget,
        "sill": sill,
        "nugget": nugget,
        "range": raw_range * prf,                   # effective range
        "nu": pdict.get(f"{comp}_nu", np.nan),
        "alpha": np.nan,
        "range_std": pstd.get(f"{comp}_range", np.nan) * (prf if np.isfinite(prf) else np.nan),
        "var_std": pstd.get(f"{comp}_sill", np.nan),
        "nugget_std": pstd.get("nugget", np.nan),
    }


In [15]:
import gstools as gs

GSTOOLS_MODELS = {"Gaussian": gs.Gaussian, "Exponential": gs.Exponential,
                  "Matern": gs.Matern, "Stable": gs.Stable, "Spherical": gs.Spherical}

def max_lag_for(path, frac=1/3):
    """Max lag = frac × domain diagonal — same rule for every library."""
    with rasterio.open(path) as src:
        b = src.bounds
    return frac * np.hypot(b.right - b.left, b.top - b.bottom)

def fit_gstools(path, sampling_size=5000, n_bins=30, percentile=0.95, seed=19920516, models=None):
    x, y, field = raster_xy(path)
    xx, yy = np.meshgrid(x, y, indexing="xy")
    pos = np.vstack([xx.ravel(), yy.ravel()]); vals = field.ravel()
    bins = np.linspace(0, max_lag_for(path), n_bins)
    bc, gamma = gs.vario_estimate(pos, vals, bins,
                                  sampling_size=sampling_size, sampling_seed=seed)
    best = None
    for name, cls in (models or GSTOOLS_MODELS).items():
        fm = cls(dim=2)
        try:
            para, pcov, r2 = fm.fit_variogram(bc, gamma, return_r2=True)
        except Exception:
            continue
        if best is None or r2 > best["pseudo_r2"]:
            _sd = dict(zip(list(para.keys()), np.sqrt(np.abs(np.diag(np.atleast_2d(pcov))))))
            best = {"method": "gstools", "best_model": name, "pseudo_r2": r2,
                    "var": fm.var, "sill": fm.sill, "nugget": fm.nugget,
                    "range": effective_range(fm, name, percentile),
                    "nu": getattr(fm, "nu", np.nan), "alpha": getattr(fm, "alpha", np.nan),
                    "range_std": float(_sd.get("len_scale", np.nan)),
                    "var_std": float(_sd.get("var", np.nan)), "nugget_std": float(_sd.get("nugget", np.nan))}
    return best

In [16]:
def fit_skg(path, fit_method="trf", sampling_size=2000, n_lags=30, seed=42, models=None):
    V = load_tif_variogram(path, sampling_size=sampling_size, n_lags=n_lags,
                           maxlag=max_lag_for(path))
    V.fit_method = fit_method
    best = None
    for name in (models or skg_models):
        try:
            V.model = name
            r2 = pseudo_r2(V)
        except Exception:
            continue
        if best is None or r2 > best["pseudo_r2"]:
            er, ps, ng, nu, al = parse_skg(V, name)
            best = {"method": "scikit-gstat", "best_model": name.capitalize(),
                    "pseudo_r2": r2, "var": ps, "sill": ps + ng, "nugget": ng,
                    "range": er, "nu": nu, "alpha": al}
    if best is not None:
        try:
            V.model = best["best_model"].lower(); _sd = np.sqrt(np.abs(np.diag(np.asarray(V.cov))))
            best["range_std"] = float(_sd[0]) if _sd.size > 0 else np.nan
            best["var_std"] = float(_sd[1]) if _sd.size > 1 else np.nan
            best["nugget_std"] = np.nan
        except Exception:
            best["range_std"] = best["var_std"] = best["nugget_std"] = np.nan
    return best


### scikit-gstat (native single-model fit)

Vanilla scikit-gstat: fit each model natively, pick by skgstat's own `rmse`, and read parameters from `V.describe()` -- a baseline against the harmonized `fit_skg`.

In [17]:
def fit_skg_native(path, fit_method="trf", sampling_size=2000, n_lags=30, models=None):
    """Single-model fit using scikit-gstat's NATIVE machinery only.

    Selects the model family by skgstat's own goodness (V.rmse) and reports
    parameters straight from V.describe() (effective_range / sill / nugget /
    smoothness|shape) -- the vanilla skgstat workflow, as a baseline alongside
    the harmonized fit_skg. (pseudo_r2 filled for the common schema; skgstat
    has no native R^2, so selection uses rmse.)"""
    V = load_tif_variogram(path, sampling_size=sampling_size, n_lags=n_lags, maxlag=max_lag_for(path))
    V.fit_method = fit_method
    best = None
    for name in (models or skg_models):
        try:
            V.model = name
            rmse = float(V.rmse)
        except Exception:
            continue
        if best is None or rmse < best[0]:
            best = (rmse, name)
    if best is None:
        return None
    _, name = best; V.model = name
    d = V.describe()
    var = float(d["sill"]); nug = float(d["nugget"])
    try:
        _sd = np.sqrt(np.abs(np.diag(np.asarray(V.cov)))); _rstd = float(_sd[0]); _vstd = float(_sd[1]) if _sd.size > 1 else np.nan
    except Exception:
        _rstd = _vstd = np.nan
    return {"method": "scikit-gstat-native", "best_model": name.capitalize(),
            "pseudo_r2": pseudo_r2(V), "var": var, "sill": var + nug, "nugget": nug,
            "range": float(d["effective_range"]),
            "nu": float(d.get("smoothness", np.nan)), "alpha": float(d.get("shape", np.nan)),
            "range_std": _rstd, "var_std": _vstd, "nugget_std": np.nan}


### Nested-model fitting

Fit multi-component (nested) variograms with each library: **topochange** (`fit_model(max_components=k)`), **gstools** (native `SumModel`), **scikit-gstat** (Hugonnet 2022-style sum of `skgstat.models` via `scipy.curve_fit`). Each returns the same contract `{method, n_components, nugget, pseudo_r2, components:[{model, range, partial_sill, nu, alpha}]}`.

In [18]:
import numpy as np
from topochange import RasterDataHandler, SingleVariogram
from topochange.variogram_models import MODEL_REGISTRY


def fit_nested_topochange(path, n_components=2):
    """Fit a nested (multi-component) variogram with topochange.

    Returns the standardized interchangeable contract:
      {"method", "n_components", "nugget", "pseudo_r2", "components": [...]}
    with each component {"model", "range" (EFFECTIVE), "partial_sill",
    "nu", "alpha"} sorted by range ascending.
    """
    # 1. read raster (PROJ_DATA override required for CRS read)
    rdh = RasterDataHandler(path, unit="m", resolution=1.0)
    rdh.load_raster(masked=True)

    # 2. empirical variogram: ~3000 pixel centres, maxlag = diagonal/3
    sv = SingleVariogram(rdh)
    sv.compute_empirical_variogram(
        area_side=1.0,
        samples_per_area=1.0,
        max_samples=3000,
        bin_width=2.0,
        max_lag_multiplier=1.0 / 3.0,
        seed=42,
        estimator="matheron",
        return_sample=True,
    )

    # 3. fit nested candidates (1..n_components) via Cressie WLS
    sv.fit_model(
        model_types=["spherical", "exponential", "matern"],
        include_nugget=True,
        max_components=n_components,
        criterion="aicc",
        seed=42,
    )

    # 4. AICc tends to under-fit the number of structures and the
    #    multi-family search can produce degenerate fits (a Matern with
    #    its smoothness nu pinned to a bound, mimicking another shape, or
    #    a component carrying ~zero sill).  Since the user fixes the
    #    component count, pick, among candidates with EXACTLY n_components
    #    bounded structures, the lowest-RSS (best-fitting) non-degenerate
    #    one.
    def n_bounded(m):
        return len(m["model"].bounded_components)

    def is_degenerate(m):
        mdl = m["model"]
        mdl.set_params(m["params"])
        total = mdl.get_stationary_sill()
        for i, name in enumerate(mdl.component_names):
            spec = MODEL_REGISTRY.get_model(name)
            cp = dict(zip(spec.param_names, mdl.get_component_params(i)))
            if spec.has_sill and total > 0 and cp.get("sill", 0.0) / total < 0.02:
                return True  # near-zero sill -> effectively fewer comps
            if "nu" in cp and (cp["nu"] >= 4.9 or cp["nu"] <= 0.11):
                return True  # Matern smoothness unidentified / overfit
        return False

    exact = [m for m in sv.fitted_models if n_bounded(m) == n_components]
    clean = [m for m in exact if not is_degenerate(m)]
    pool = clean if clean else exact
    best = min(pool, key=lambda m: m["rss"]) if pool else sv.best_model

    model = best["model"]
    model.set_params(best["params"])
    nugget = float(model.get_nugget())

    # 5. extract components; convert raw range -> EFFECTIVE range via the
    #    model's practical_range_factor (1.0 for spherical and for the
    #    sqrt(2*nu) Matern parameterization, 3.0 for exponential).
    components = []
    for i, name in enumerate(model.component_names):
        spec = MODEL_REGISTRY.get_model(name)
        cp = dict(zip(spec.param_names, model.get_component_params(i)))
        raw_range = float(cp.get("range", np.nan))
        prf = spec.practical_range_factor
        eff_range = (raw_range * prf
                     if (prf is not None and np.isfinite(raw_range))
                     else raw_range)
        components.append({
            "model": name,
            "range": float(eff_range),
            "partial_sill": float(cp.get("sill", np.nan)),
            "nu": float(cp["nu"]) if "nu" in cp else None,
            "alpha": None,
        })
    components.sort(key=lambda c: c["range"])

    # 6. pseudo R2 of the fitted SUM model vs the empirical variogram
    gamma_pred = model(sv.lags)
    gamma_obs = sv.variogram
    ss_res = float(np.sum((gamma_obs - gamma_pred) ** 2))
    ss_tot = float(np.sum((gamma_obs - np.mean(gamma_obs)) ** 2))
    pseudo_r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return {
        "method": "topochange",
        "n_components": int(len(model.component_names)),
        "nugget": nugget,
        "pseudo_r2": float(pseudo_r2),
        "components": components,
    }

In [19]:
import numpy as np
import rasterio
import gstools as gs

# Models whose variogram reaches the sill exactly at len_scale
# (finite / practical range == len_scale). All others are asymptotic.
_NESTED_FINITE = {"Spherical", "Circular", "Cubic", "Linear", "HyperSpherical", "Intersection"}


def _effective_range(model):
    """Effective (practical) range of a single gstools CovModel.
    Finite-range models: len_scale. Asymptotic models: 95% practical range."""
    if model.name in _NESTED_FINITE:
        return float(model.len_scale)
    return float(model.percentile_scale(0.95))


def _empirical_variogram(path, n_sample=3000, n_bins=60, seed=42):
    """Read raster, subsample ~n_sample valid pixel centres, build empirical variogram
    with maxlag = (domain diagonal)/3. Returns (bin_center, gamma, maxlag)."""
    with rasterio.open(path) as ds:
        arr = ds.read(1).astype(float)
        nod = ds.nodata
        transform = ds.transform
        ny, nx = arr.shape
    cols, rows = np.meshgrid(np.arange(nx), np.arange(ny))
    xs, ys = rasterio.transform.xy(transform, rows.ravel(), cols.ravel())
    xs = np.asarray(xs); ys = np.asarray(ys)
    vals = arr.ravel()
    mask = np.isfinite(vals)
    if nod is not None:
        mask &= (vals != nod)
    xs, ys, vals = xs[mask], ys[mask], vals[mask]
    rng = np.random.default_rng(seed)
    n = min(n_sample, vals.size)
    idx = rng.choice(vals.size, size=n, replace=False)
    xs, ys, vals = xs[idx], ys[idx], vals[idx]

    diag = float(np.hypot(xs.max() - xs.min(), ys.max() - ys.min()))
    maxlag = diag / 3.0
    bin_edges = np.linspace(0, maxlag, n_bins + 1)
    bin_center, gamma = gs.vario_estimate((xs, ys), vals, bin_edges=bin_edges)
    ok = np.isfinite(gamma)
    return bin_center[ok], gamma[ok], maxlag


def _build_sum_model(n_components, maxlag, mix=False):
    """Construct an (unfitted) gstools SumModel of n_components.
    Each sub-model's len_scale is bounded to (0.1, maxlag] (open upper) so no component
    runs away to a spurious huge range that mimics a linear drift / trend.
    mix=True (only for n=2) builds Matern(nu free) + Spherical."""
    if mix and n_components == 2:
        subs = [gs.Matern(dim=2), gs.Spherical(dim=2)]
    else:
        subs = [gs.Spherical(dim=2) for _ in range(n_components)]
    for s in subs:
        s.len_scale_bounds = [0.1, maxlag, "co"]  # 'co' = closed-open avoids edge errors
    model = subs[0]
    for s in subs[1:]:
        model = model + s  # '+' on CovModels builds a gs.SumModel
    return model


def fit_nested_gstools(path, n_components=2):
    """Fit a nested (multi-component) variogram with gstools SumModel.
    Returns the interchangeable contract dict (components sorted by range ascending)."""
    bin_center, gamma, maxlag = _empirical_variogram(path)

    best = None
    trials = [False]
    if n_components == 2:
        trials.append(True)  # also try Matern+Spherical mixture, keep whichever fits best
    for mix in trials:
        model = _build_sum_model(n_components, maxlag, mix=mix)
        try:
            # gstools return_r2 == 1 - SSres/SStot of the fitted SUM model vs empirical
            _, _, r2 = model.fit_variogram(bin_center, gamma, nugget=True, return_r2=True)
        except Exception:
            continue
        if best is None or r2 > best[0]:
            best = (r2, model)
    r2, model = best

    # SumModel exposes its fitted sub-models via .models (list of CovModel);
    # per-submodel partial sill is sub.var, length scale is sub.len_scale;
    # the combined nugget is model.nugget; total sill is model.sill.
    comps = []
    for sub in model.models:
        nu = float(sub.nu) if "nu" in sub.opt_arg else None
        alpha = float(sub.alpha) if "alpha" in sub.opt_arg else None
        comps.append({
            "model": sub.name,
            "range": _effective_range(sub),      # EFFECTIVE range
            "partial_sill": float(sub.var),
            "nu": nu,
            "alpha": alpha,
        })
    comps.sort(key=lambda c: c["range"])

    return {
        "method": "gstools",
        "n_components": int(n_components),
        "nugget": float(model.nugget),
        "pseudo_r2": float(r2),
        "components": comps,
    }

In [20]:
import numpy as np
import rasterio
import skgstat as skg
from skgstat import models as _skg_modfns
from scipy.optimize import curve_fit


def _build_empirical(path, n_sample=3000, n_lags=50, seed=42):
    """Read raster, subsample ~n_sample pixel centres, build empirical variogram.

    maxlag = (domain diagonal) / 3. Returns
    (bins, experimental, counts, bin_width, maxlag, total_var).
    """
    with rasterio.open(path) as ds:
        arr = ds.read(1).astype(float)
        res_x, res_y = ds.res
        nrows, ncols = arr.shape

    rr, cc = np.where(np.isfinite(arr))
    vals = arr[rr, cc]
    xs = (cc + 0.5) * res_x
    ys = (rr + 0.5) * res_y

    rng = np.random.default_rng(seed)
    n = len(vals)
    if n > n_sample:
        idx = rng.choice(n, size=n_sample, replace=False)
        xs, ys, vals = xs[idx], ys[idx], vals[idx]

    coords = np.column_stack([xs, ys])
    diag = np.hypot(ncols * res_x, nrows * res_y)
    maxlag = diag / 3.0

    V = skg.Variogram(
        coords, vals, maxlag=maxlag, n_lags=n_lags,
        estimator="matheron", normalize=False, fit_method=None,
    )
    bins = np.asarray(V.bins, dtype=float)
    exp = np.asarray(V.experimental, dtype=float)
    counts = np.asarray(V.bin_count, dtype=float)
    bin_width = bins[0] if len(bins) else maxlag / n_lags
    total_var = float(np.var(vals))

    good = np.isfinite(exp) & (counts > 0)
    return (bins[good], exp[good], counts[good],
            float(bin_width), float(maxlag), total_var)


def fit_nested_skgstat(path, n_components=2, n_sample=3000, n_lags=50, seed=42):
    """Fit a nested (multi-component) variogram with scikit-gstat (Hugonnet et al. 2022 style).

    scikit-gstat 1.0.22 has NO built-in sum/nested model, so we build the
    empirical variogram with skgstat.Variogram (V.bins / V.experimental) and fit
    a SUM of skgstat.models functions with scipy.optimize.curve_fit. Residuals are
    weighted by 1/sqrt(pair count) (Hugonnet-style) so well-populated bins matter
    more, and multi-start is used for identifiability. The all-spherical sum is the
    primary model; for >=2 components a matern(short)+spherical(rest) alternative
    is also tried and adopted only if it improves the weighted SSE by a clear
    margin (the extra Matern smoothness parameter must earn its keep, otherwise
    genuinely spherical nested structures get overfit).

    "range" is the EFFECTIVE/practical range. The skgstat `r` argument IS defined
    as the effective range for every model used here: spherical r == effective
    range (== len_scale); matern r == the lag at which 95% of the sill is reached.
    So `r` is returned directly. Components are sorted by range ascending.
    """
    bins, exp, counts, bin_width, maxlag, total_var = _build_empirical(
        path, n_sample=n_sample, n_lags=n_lags, seed=seed
    )

    sph = _skg_modfns.spherical   # signature: (h, r, c0, b=0.0)
    mat = _skg_modfns.matern      # signature: (h, r, c0, s, b=0.0)

    def sph_vec(h, r, c0):
        return sph(h, r, c0, 0.0)

    def mat_vec(h, r, c0, s):
        return mat(h, r, c0, s, 0.0)

    sigma = 1.0 / np.sqrt(np.maximum(counts, 1.0))
    w = counts / counts.sum()
    exp_wmean = np.sum(w * exp)
    sstot = float(np.sum((exp - exp_wmean) ** 2))

    lo_r = max(bin_width, 1e-6)
    hi_r = maxlag

    def range_seeds(k, jitter=1.0):
        seeds = np.geomspace(lo_r * 1.5, hi_r * 0.8, k)
        return np.clip(seeds * jitter, lo_r * 1.01, hi_r * 0.99)

    def make_sph_sum(k):
        def model(h, *p):
            nug = p[0]
            out = np.full_like(np.asarray(h, dtype=float), nug)
            for i in range(k):
                out = out + sph_vec(h, p[1 + 2 * i], p[2 + 2 * i])
            return out
        return model

    def _fit(model, p0, bounds):
        try:
            popt, _ = curve_fit(model, bins, exp, p0=p0, bounds=bounds,
                                sigma=sigma, absolute_sigma=False, maxfev=300000)
            pred = model(bins, *popt)
            sse = float(np.sum((exp - pred) ** 2))
            return sse, popt
        except Exception:
            return None

    k = n_components

    # ---- all-spherical SUM model (multi-start) ----
    best_sph = None
    model_sph = make_sph_sum(k)
    lb = [0.0] + [lo_r, 0.0] * k
    ub = [total_var * 1.2 + 1e-9] + [hi_r, total_var * 2 + 1e-9] * k
    sill_seed = max(total_var / k, 1e-3)
    for jit in [1.0, 0.6, 1.6, 0.35, 2.2, 0.8, 1.2]:
        rs = range_seeds(k, jit)
        p0 = [max(total_var * 0.02, 1e-4)]
        for i in range(k):
            p0 += [float(rs[i]), sill_seed]
        r = _fit(model_sph, p0, (lb, ub))
        if r is not None and (best_sph is None or r[0] < best_sph[0]):
            best_sph = r

    # ---- matern(short) + spherical(rest) ALTERNATIVE ----
    best_ms = None
    if k >= 2:
        def make_mat_sph_sum(k):
            def model(h, *p):
                nug = p[0]
                out = np.full_like(np.asarray(h, dtype=float), nug)
                out = out + mat_vec(h, p[1], p[2], p[3])   # matern short component
                base = 4
                for i in range(k - 1):
                    out = out + sph_vec(h, p[base + 2 * i], p[base + 1 + 2 * i])
                return out
            return model

        model_ms = make_mat_sph_sum(k)
        # nu bounded to a typical, well-behaved range. Beyond ~2.5 the Matern
        # curvature is nearly indistinguishable from other models and nu is only
        # weakly identifiable, so the optimiser drifts to the bound for <1% SSE
        # gain; capping keeps nu physically meaningful.
        lb_ms = [0.0, lo_r, 0.0, 0.3] + [lo_r, 0.0] * (k - 1)
        ub_ms = ([total_var * 1.2 + 1e-9, hi_r, total_var * 2 + 1e-9, 2.5]
                 + [hi_r, total_var * 2 + 1e-9] * (k - 1))
        for jit in [1.0, 0.6, 1.6, 0.35]:
            rs = range_seeds(k, jit)
            for nu0 in (1.5, 0.5):
                p0 = [max(total_var * 0.02, 1e-4), float(rs[0]), sill_seed, nu0]
                for i in range(k - 1):
                    p0 += [float(rs[i + 1]), sill_seed]
                r = _fit(model_ms, p0, (lb_ms, ub_ms))
                if r is not None and (best_ms is None or r[0] < best_ms[0]):
                    best_ms = r

    # ---- model selection: prefer all-spherical unless matern clearly wins ----
    use_matern = False
    if best_ms is not None and best_sph is not None:
        if best_ms[0] < 0.92 * best_sph[0]:
            use_matern = True
    elif best_ms is not None and best_sph is None:
        use_matern = True

    if use_matern:
        sse, popt = best_ms
        kind = "matern_spherical"
    else:
        sse, popt = best_sph
        kind = "spherical_sum"

    pseudo_r2 = 1.0 - sse / sstot if sstot > 0 else float("nan")

    comps = []
    if kind == "spherical_sum":
        nugget = float(popt[0])
        for i in range(k):
            comps.append({"model": "spherical", "range": float(popt[1 + 2 * i]),
                          "partial_sill": float(popt[2 + 2 * i]), "nu": None, "alpha": None})
    else:
        nugget = float(popt[0])
        comps.append({"model": "matern", "range": float(popt[1]),
                      "partial_sill": float(popt[2]), "nu": float(popt[3]), "alpha": None})
        base = 4
        for i in range(k - 1):
            comps.append({"model": "spherical", "range": float(popt[base + 2 * i]),
                          "partial_sill": float(popt[base + 1 + 2 * i]), "nu": None, "alpha": None})

    comps.sort(key=lambda c: c["range"])

    return {"method": "skgstat", "n_components": int(k), "nugget": float(nugget),
            "pseudo_r2": float(pseudo_r2), "components": comps}

### Run fitting over the selection

In [22]:
# ===== run the fitting comparison over SELECTION (single + nested, unified) =====
SINGLE_FITTERS = {"topochange": fit_topochange, "gstools": fit_gstools,
                  "scikit-gstat": fit_skg, "scikit-gstat-native": fit_skg_native}
NESTED_FITTERS = {"topochange": fit_nested_topochange, "gstools": fit_nested_gstools,
                  "scikit-gstat": fit_nested_skgstat}

def _truth_components(truth):
    if truth.get("components"):
        return [dict(c) for c in truth["components"]]
    return [{"model": truth["best_model"], "range": truth["range"], "partial_sill": truth["var"],
             "nu": truth.get("nu"), "alpha": truth.get("alpha")}]

def _as_components(res, kind):
    if kind == "nested":
        return sorted(res.get("components", []), key=lambda c: c["range"])
    return [{"model": res["best_model"], "range": res["range"], "partial_sill": res["var"],
             "nu": res.get("nu"), "alpha": res.get("alpha")}]

def score_file(row):
    """Fit one raster (single or nested) with all relevant libraries -> long scored rows."""
    path, kind = row["path"], row["kind"]
    if kind not in ("single", "nested"):   # nugget/syserr/trend/aniso: generation only for now
        return []
    truth = truth_from_tags(path)
    tcomps = sorted(_truth_components(truth),
                    key=lambda c: (c["range"] if np.isfinite(c["range"]) else 1e18))
    base = dict(file=row["file"], kind=kind, n_true=len(tcomps),
                separation_ratio=row.get("separation_ratio", np.nan), true_nugget=truth["nugget"])
    fitters = NESTED_FITTERS if kind == "nested" else SINGLE_FITTERS
    rows = []
    for mname, fn in fitters.items():
        try:
            res = fn(path, len(tcomps)) if kind == "nested" else fn(path)
            rcomps = _as_components(res, kind); rnug = res.get("nugget", np.nan); r2 = res.get("pseudo_r2", np.nan)
        except Exception as e:
            rows.append({**base, "method": mname, "comp": np.nan, "error": str(e)[:90]}); continue
        for i, tc in enumerate(tcomps):
            rc = rcomps[i] if i < len(rcomps) else None
            rows.append({**base, "method": mname, "comp": i + 1, "pseudo_r2": r2, "rec_nugget": rnug,
                         "true_model": tc["model"], "true_range": tc["range"], "true_sill": tc["partial_sill"],
                         "rec_model": (rc["model"] if rc else None),
                         "rec_range": (rc["range"] if rc else np.nan),
                         "rec_sill": (rc["partial_sill"] if rc else np.nan),
                         "range_relerr": ((rc["range"] - tc["range"]) / tc["range"]
                                          if (rc and tc["range"]) else np.nan),
                         "sill_err": (rc["partial_sill"] - tc["partial_sill"] if rc else np.nan),
                         "model_correct": (rc is not None and str(rc["model"]).lower() == str(tc["model"]).lower())})
    return rows

def run_fits(selection, out_csv=None, progress=25):
    recs = []
    for i, (_, row) in enumerate(selection.iterrows()):
        recs += score_file(row)
        if progress and (i + 1) % progress == 0:
            print(f"  fit [{i+1}/{len(selection)}]")
    df = pd.DataFrame(recs)
    if out_csv:
        df.to_csv(out_csv, index=False); print(f"saved {out_csv} ({len(df)} rows)")
    return df

RUN_FITS = True     # <-- set True to fit the whole SELECTION
if RUN_FITS:
    fits_df = run_fits(SELECTION, out_csv=os.path.join(DATA_DIR, "fits_comparison.csv"))
    display(fits_df.head(20))
elif DEV_PATH:       # otherwise: quick single-file preview
    display(pd.DataFrame(score_file(SELECTION.iloc[0])))

NameError: name 'truth_from_tags' is not defined

## Areal uncertainties

Propagate each fitted model (truth + every library) to the areal-mean uncertainty over the polygons, for the rasters in `SELECTION`.

In [12]:
import numpy as np, math, shapely, rasterio

def cov_from_model(m):
    """C(h) = sigma2 - gamma(h) from a builder dict {'gamma','sigma2',...}."""
    s2 = m["sigma2"]
    return lambda h: s2 - np.asarray(m["gamma"](np.asarray(h, float)), float)

def poly_grid(path, poly):
    """Pixel-center coords of a raster that fall inside poly, plus mask & resolution."""
    with rasterio.open(path) as src:
        t, W, H = src.transform, src.width, src.height
    xs = t.c + t.a * (np.arange(W) + 0.5)
    ys = t.f + t.e * (np.arange(H) + 0.5)
    gx, gy = np.meshgrid(xs, ys)
    mask = shapely.contains_xy(poly, gx, gy)         # (H, W)
    pts = np.column_stack([gx[mask], gy[mask]])
    return pts, mask, abs(t.a)

### FFT (autocorrelation of the polygon mask) 
def unc_fft(model, poly, path):
    """Same 1^T C 1, but via FFT of the polygon mask -> O(N log N)."""
    cov = cov_from_model(model)
    _, mask, res = poly_grid(path, poly)
    H, W = mask.shape
    N = mask.sum()
    fy, fx = 2 * H, 2 * W                            # pad to get linear autocorrelation
    F = np.fft.rfft2(mask.astype(float), s=(fy, fx))
    ac = np.fft.irfft2(F * np.conj(F), s=(fy, fx))   # ac[dy,dx] = # pixel pairs at that lag
    dy = np.fft.fftfreq(fy) * fy
    dx = np.fft.fftfreq(fx) * fx
    DY, DX = np.meshgrid(dy, dx, indexing="ij")
    dist = res * np.hypot(DY, DX)
    v = float((ac * cov(dist)).sum() / (N * N))
    return math.sqrt(v) if v > 0 else 0.0

In [ ]:
### point sampler 
import numpy as np, shapely

def _sample_in(poly, n, rng):
    """Uniformly sample n points inside a polygon (rejection sampling)."""
    minx, miny, maxx, maxy = poly.bounds
    xs, ys = np.empty(0), np.empty(0)
    while xs.size < n:
        k = int((n - xs.size) * 1.5) + 100
        rx, ry = rng.uniform(minx, maxx, k), rng.uniform(miny, maxy, k)
        m = shapely.contains_xy(poly, rx, ry)
        xs, ys = np.concatenate([xs, rx[m]]), np.concatenate([ys, ry[m]])
    return xs[:n], ys[:n]


In [ ]:
### Monte-Carlo pair integral 
def unc_mc(model, poly, n_pairs=100_000, seed=42):
    """sqrt(mean C(h)) over random point pairs in the polygon (continuous integral)."""
    cov = cov_from_model(model)
    rng = np.random.default_rng(seed)
    x1, y1 = _sample_in(poly, n_pairs, rng)          # _sample_in from Cell A
    x2, y2 = _sample_in(poly, n_pairs, rng)
    v = float(np.mean(cov(np.hypot(x1 - x2, y1 - y2))))
    return math.sqrt(v) if v > 0 else 0.0

### Exact discretized matrix sum (chunked 1ᵀC1)
def unc_matrix(model, poly, path, block=2000):
    """Exact discrete block variance (1/N^2) sum_ij C(|ci-cj|) over polygon pixels."""
    cov = cov_from_model(model)
    pts, _, _ = poly_grid(path, poly)
    N = len(pts)
    total = 0.0
    for s in range(0, N, block):                     # chunk rows to bound memory
        a = pts[s:s + block]
        d = np.sqrt(((a[:, None, :] - pts[None, :, :]) ** 2).sum(-1))
        total += cov(d).sum()
    v = total / (N * N)
    return math.sqrt(v) if v > 0 else 0.0

### Integral range / effective sample size (Rolstad/Hugonnet)
def integral_range_2d(model, rmax=None, n=4000, thresh=1e-3):
    """S = ∫ rho(h) 2*pi*h dh, rho = C/sigma2 (2-D integral range)."""
    cov, s2 = cov_from_model(model), model["sigma2"]
    if rmax is None:                                 # integrate to where correlation ~ 0
        rs = np.linspace(0, 5000, n); rho = cov(rs) / s2
        below = np.where(rho < thresh)[0]
        rmax = rs[below[0]] if below.size else 5000.0
    r = np.linspace(0, rmax, n)
    return float(np.trapezoid((cov(r) / s2) * 2 * np.pi * r, r))

def unc_integral_range(model, poly):
    """Asymptotic Var(mean) = sigma2 * S / A  (valid when A >> S; shape-blind)."""
    S = integral_range_2d(model)
    v = model["sigma2"] * S / poly.area
    return (math.sqrt(v) if v > 0 else 0.0), S, poly.area / S


In [ ]:
### block resampling (model-free, uses the actual field)
def unc_block_resample(poly, path, n_place=400, overlap=True, seed=0):
    """Std of the polygon-shaped areal mean across many placements on the field."""
    x, y, field = raster_xy(path)
    _, mask, _ = poly_grid(path, poly)               # polygon shape in pixels
    # trim mask to its tight bbox so it slides over the field
    rows = np.any(mask, axis=1); cols = np.any(mask, axis=0)
    mask = mask[np.ix_(rows, cols)]
    wy, wx = mask.shape
    H, W = field.shape
    rng = np.random.default_rng(seed)
    means = []
    if overlap:                                      # many random placements (correlated)
        for _ in range(n_place):
            r0, c0 = rng.integers(0, H - wy + 1), rng.integers(0, W - wx + 1)
            sel = field[r0:r0 + wy, c0:c0 + wx][mask]
            if np.isfinite(sel).all():
                means.append(sel.mean())
    else:                                            # non-overlapping tiles (independent, few)
        for r0 in range(0, H - wy + 1, wy):
            for c0 in range(0, W - wx + 1, wx):
                sel = field[r0:r0 + wy, c0:c0 + wx][mask]
                if np.isfinite(sel).all():
                    means.append(sel.mean())
    means = np.asarray(means)
    return float(means.std(ddof=1)), len(means)


In [ ]:
### simulation-based (GRF realizations → std of areal means)
def unc_simulation(gs_model, poly, path, n_real=200, pad=2, seed=0):
    """Gold-standard ensemble std of the areal mean from circulant-embedding sims.
    gs_model: a gstools CovModel (e.g. gs.Spherical(var=sill, len_scale=range))."""
    _, mask, res = poly_grid(path, poly)
    shape = mask.shape
    means = [grf_circulant(gs_model, shape, res, seed=seed + k, pad=pad)[mask].mean()
             for k in range(n_real)]
    return float(np.std(means, ddof=1))


In [ ]:
### Closed-form (analytic) for disk / rectangle  -- isotropic C only
import warnings as _warnings
def _mrr(poly):
    """minimum_rotated_rectangle with shapely's benign oriented_envelope warnings suppressed."""
    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore")
        return poly.minimum_rotated_rectangle

# 4-D block integral collapses via the shape's geometric covariogram (set-overlap).
_trap = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

def unc_disk_analytic(model, poly, n=6000):
    """sigma_A for a disk via the lens-overlap kernel (1-D integral)."""
    cov = cov_from_model(model)
    R = math.sqrt(poly.area / math.pi); A = math.pi * R ** 2
    r = np.linspace(0, 2 * R, n)
    K = 2 * R ** 2 * np.arccos(np.clip(r / (2 * R), 0., 1.)) - (r / 2) * np.sqrt(np.clip(4 * R ** 2 - r ** 2, 0., None))
    v = (2 * np.pi / A ** 2) * _trap(K * cov(r) * r, r)
    return math.sqrt(v) if v > 0 else 0.0

def unc_rect_analytic(model, poly, n=800):
    """sigma_A for a rectangle via the triangular set-covariogram kernel (2-D integral).
    Side lengths from the oriented bbox; orientation is irrelevant for isotropic C."""
    cov = cov_from_model(model)
    pts = np.array(_mrr(poly).exterior.coords)[:-1]
    d = np.sort(np.sqrt(((np.roll(pts, -1, 0) - pts) ** 2).sum(1)))
    a, b = d[0], d[2]; A = a * b
    u = np.linspace(0, a, n); v = np.linspace(0, b, n)
    U, V = np.meshgrid(u, v, indexing="ij")
    integ = (a - U) * (b - V) * cov(np.hypot(U, V))
    val = 4 / A ** 2 * _trap(_trap(integ, v, axis=1), u, axis=0)
    return math.sqrt(val) if val > 0 else 0.0

def unc_analytic(model, poly, path=None):
    """Closed-form sigma_A: auto-detect disk vs rectangle from the bbox fill ratio.
    Disk fills pi/4 (~0.785) of its bounding box; a rectangle fills ~1.
    Falls back to exact FFT for irregular polygons (needs path=)."""
    fill = poly.area / _mrr(poly).area
    if fill > 0.95:
        return unc_rect_analytic(model, poly)
    if 0.70 < fill < 0.85:
        return unc_disk_analytic(model, poly)
    if path is not None:
        return unc_fft(model, poly, path)        # irregular -> exact numerical
    raise ValueError(f"polygon is neither disk nor rectangle (fill={fill:.3f}); pass path= for FFT fallback")


In [ ]:
### Areal model builder: polygons, circulant simulator, effective-range->len_scale, model_from_row
import os, geopandas as gpd

# areas of interest (same CRS/units as the rasters)
poly_paths = {
    "circle":    "synthetic_benchmark/polygons_2/circle1.shp",
    "rectangle": "synthetic_benchmark/polygons_2/rectangle1.shp",
    "square":    "synthetic_benchmark/polygons_2/square1.shp",
}
poly_geoms = {k: gpd.read_file(v).union_all() for k, v in poly_paths.items()}

# exact GRF simulator (used by unc_simulation)
def grf_circulant(model, shape, resolution=1.0, seed=None, pad=2):
    ny, nx = shape
    My, Mx = pad * ny, pad * nx
    ky = np.concatenate([np.arange(0, My // 2 + 1), np.arange(My // 2 + 1 - My, 0)])
    kx = np.concatenate([np.arange(0, Mx // 2 + 1), np.arange(Mx // 2 + 1 - Mx, 0)])
    Y, X = np.meshgrid(ky * resolution, kx * resolution, indexing="ij")
    lam = np.fft.fft2(model.covariance(np.hypot(X, Y))).real
    lam[lam < 0] = 0.0
    r = np.random.default_rng(seed)
    xi = r.normal(size=(My, Mx)) + 1j * r.normal(size=(My, Mx))
    return (np.fft.fft2(xi * np.sqrt(lam)) / np.sqrt(My * Mx)).real[:ny, :nx]

# effective range -> len_scale (inverse of the generation convention)
def len_scale_for_range(model_cls, dim, target_range, percentile=0.95, **kw):
    if model_cls.__name__ in _FINITE_RANGE:
        return float(target_range)
    unit = model_cls(dim=dim, var=1.0, len_scale=1.0, **kw)
    return float(target_range) / unit.percentile_scale(percentile)

def model_from_row(row, dim=2):
    """Reconstruct a gstools model (+ gamma/sigma2) from a comparison-table row."""
    comps = row.get("components") if hasattr(row, "get") else None
    if comps is not None and len(comps) > 0:                 # nested -> gstools SumModel
        sm = None
        for c in comps:
            cn = c["model"]; ccls = getattr(gs, cn if hasattr(gs, cn) else cn.capitalize())
            ex = {}
            if c.get("nu") is not None and np.isfinite(float(c["nu"])): ex["nu"] = float(c["nu"])
            if c.get("alpha") is not None and np.isfinite(float(c["alpha"])): ex["alpha"] = float(c["alpha"])
            sub = ccls(dim=dim, var=float(c["partial_sill"]),
                       len_scale=len_scale_for_range(ccls, dim, float(c["range"]), percentile, **ex), **ex)
            sm = sub if sm is None else sm + sub
        sm.nugget = float(row.get("nugget", 0.0) or 0.0)
        return {"method": row.get("method", "?"), "family": f"nested{len(comps)}", "gs_model": sm,
                "gamma": lambda h, sm=sm: sm.variogram(np.asarray(h, float)),
                "sigma2": float(sm.sill), "nugget": float(sm.nugget)}
    name = row["best_model"]
    cls = getattr(gs, name)
    extra = {}
    if name == "Matern" and np.isfinite(row.get("nu", np.nan)):
        extra["nu"] = float(row["nu"])
    if name == "Stable" and np.isfinite(row.get("alpha", np.nan)):
        extra["alpha"] = float(row["alpha"])
    ls = len_scale_for_range(cls, dim, float(row["range"]), percentile, **extra)
    fm = cls(dim=dim, var=float(row["var"]), len_scale=ls, nugget=float(row["nugget"]), **extra)
    return {"method": row["method"], "family": name, "gs_model": fm,
            "gamma": lambda h, fm=fm: fm.variogram(np.asarray(h, float)),
            "sigma2": float(fm.sill), "nugget": float(row["nugget"])}


### Run areal uncertainty over the selection

In [ ]:
# ===== areal uncertainty over SELECTION (truth + every fitted model) =====
AREAL_METHODS = {
    "MC":             lambda m, g, p: unc_mc(m, g),
    "matrix":         lambda m, g, p: unc_matrix(m, g, p),
    "fft":            lambda m, g, p: unc_fft(m, g, p),
    "analytic":       lambda m, g, p: unc_analytic(m, g, p),
    "integral_range": lambda m, g, p: unc_integral_range(m, g)[0],
    "simulation":     lambda m, g, p: unc_simulation(m["gs_model"], g, p, n_real=100),
}

def _row_for_mfr(res, kind, method):
    if kind == "nested":
        return {"method": method, "components": res["components"], "nugget": res["nugget"]}
    return {"method": method, "best_model": res["best_model"], "var": res["var"], "sill": res["sill"],
            "nugget": res["nugget"], "range": res["range"], "nu": res.get("nu"), "alpha": res.get("alpha")}

def models_for_file(path, kind):
    """{param_source: model_dict} for truth + each fitting library (via model_from_row)."""
    truth = truth_from_tags(path)
    out = {"truth": model_from_row(truth)}
    k = len(truth["components"]) if truth.get("components") else 1
    fitters = NESTED_FITTERS if kind == "nested" else SINGLE_FITTERS
    for mname, fn in fitters.items():
        try:
            res = fn(path, k) if kind == "nested" else fn(path)
            out[mname] = model_from_row(_row_for_mfr(res, kind, mname))
        except Exception:
            out[mname] = None
    return out

def areal_for_file(row, area_method="fft", polygons=None):
    polygons = polygons or poly_geoms
    path = row["path"]; rows = []
    for src, m in models_for_file(path, row["kind"]).items():
        if m is None:
            continue
        for pname, geom in polygons.items():
            rows.append({"file": row["file"], "kind": row["kind"], "param_source": src,
                         "family": m["family"], "polygon": pname, "area_method": area_method,
                         "sigma2": m["sigma2"], "sigma_A": AREAL_METHODS[area_method](m, geom, path)})
    return rows

def run_areal(selection, area_method="fft", out_csv=None, progress=10):
    sel = selection[selection["kind"] != "nugget"]; recs = []
    for i, (_, row) in enumerate(sel.iterrows()):
        recs += areal_for_file(row, area_method=area_method)
        if progress and (i + 1) % progress == 0:
            print(f"  areal [{i+1}/{len(sel)}]")
    df = pd.DataFrame(recs)
    if out_csv:
        df.to_csv(out_csv, index=False); print(f"saved {out_csv} ({len(df)} rows)")
    return df

def areal_methods_table(path, param_source="truth"):
    """Deep dive: every area method x polygon for one file's chosen param source."""
    kind = parse_name(os.path.basename(path))["kind"]
    m = models_for_file(path, kind).get(param_source)
    rows = [{"area_method": am, **{pn: AREAL_METHODS[am](m, g, path) for pn, g in poly_geoms.items()}}
            for am in AREAL_METHODS]
    return pd.DataFrame(rows).set_index("area_method").round(5)

RUN_AREAL = False    # <-- set True to compute areal uncertainty over the whole SELECTION
if RUN_AREAL:
    areal_df = run_areal(SELECTION, area_method="fft", out_csv=os.path.join(DATA_DIR, "areal_comparison.csv"))
    display(areal_df.head(20))
elif DEV_PATH:       # otherwise: deep-dive one file (all area methods x polygons, truth model)
    display(areal_methods_table(DEV_PATH, param_source="truth"))

## Stress-test scoring

Staged evaluation over the `catalog`. **Tier 0** (below) is cheap whole-field + block statistics; fitting/areal tiers follow. `EVAL_SELECTION` is the stratified n=1000 subsample used for the expensive inter-package comparisons.

In [ ]:
### Evaluation subsample (stratified, n=1000)
# Stratify over the discrete axes so every level is represented; uniform within cells covers the rest.
def stratified_sample(cat, n=1000, strat=("kind", "model", "nugget_frac"), seed=42):
    strat = [c for c in strat if c in cat.columns]
    g = cat.groupby(strat, dropna=False)
    per = max(1, n // max(g.ngroups, 1))
    idx = []
    for _, d in g:
        idx += list(d.sample(min(len(d), per), random_state=seed).index)
    samp = cat.loc[idx]
    if len(samp) > n:
        samp = samp.sample(n, random_state=seed)
    elif len(samp) < n:                                  # top up randomly from the remainder
        rest = cat.loc[~cat.index.isin(idx)]
        if len(rest):
            samp = pd.concat([samp, rest.sample(min(len(rest), n - len(samp)), random_state=seed)])
    return samp.reset_index(drop=True)

EVAL_SELECTION = stratified_sample(catalog, n=1000) if len(catalog) else catalog
print(f"stratified eval subsample: {len(EVAL_SELECTION)} files")
if len(EVAL_SELECTION):
    print(EVAL_SELECTION.groupby("kind").size().to_string())
    for c in ("model", "nugget_frac"):
        if c in EVAL_SELECTION.columns:
            print(f"  {c}: {EVAL_SELECTION[c].value_counts(dropna=False).to_dict()}")
EVAL_SELECTION.head()

In [ ]:
### Tier 0 -- statistics (whole-field + block); cheap, run on any selection
import rasterio

def _field_stats_arr(arr):
    finite = np.isfinite(arr); v = arr[finite]
    out = {"mean": float(v.mean()), "median": float(np.median(v)),
           "std": float(v.std()), "var": float(v.var())}
    for k in (2, 3):                                    # k x k block means
        rsp = np.array_split(np.arange(arr.shape[0]), k); csp = np.array_split(np.arange(arr.shape[1]), k)
        bm, cx, cy = [], [], []
        for ri in rsp:
            for ci in csp:
                bm.append(float(np.nanmean(arr[np.ix_(ri, ci)]))); cy.append(ri.mean()); cx.append(ci.mean())
        bm = np.array(bm)
        out[f"blk{k}x{k}_mean_var"] = float(np.nanvar(bm))
        out[f"blk{k}x{k}_mean_range"] = float(np.nanmax(bm) - np.nanmin(bm))
        if k == 3:                                       # planar fit of the 3x3 block means -> trend
            A = np.column_stack([np.array(cx), np.array(cy), np.ones(9)])
            coef, *_ = np.linalg.lstsq(A, bm, rcond=None)
            out["blk_trend_slope"] = float(np.hypot(coef[0], coef[1]))     # per-pixel gradient
            out["blk_trend_dir"] = float(np.degrees(np.arctan2(coef[1], coef[0])))
    return out

def field_stats(path):
    with rasterio.open(path) as s:
        return _field_stats_arr(s.read(1).astype(float))

def run_stats(selection, out_csv=None, progress=500):
    meta_cols = [c for c in ("file", "path", "kind", "model", "range", "sill",
                             "nugget_frac", "separation_ratio", "syserr", "slope",
                             "ani_ratio", "ani_major") if c in selection.columns]
    recs = []
    for i, (_, row) in enumerate(selection.iterrows()):
        rec = {c: row[c] for c in meta_cols}
        rec.update(field_stats(row["path"]))
        recs.append(rec)
        if progress and (i + 1) % progress == 0:
            print(f"  stats [{i+1}/{len(selection)}]")
    df = pd.DataFrame(recs)
    if out_csv:
        df.to_csv(out_csv, index=False); print(f"saved {out_csv} ({len(df)} rows)")
    return df

STATS_SELECTION = catalog          # cheap -> default the whole catalog (edit to subsample)
RUN_STATS = False                  # <-- set True to compute stats over STATS_SELECTION
if RUN_STATS:
    stats_df = run_stats(STATS_SELECTION, out_csv=os.path.join(DATA_DIR, "stats.csv"))
    display(stats_df.head(20))

### Tier 2 -- variogram fitting (resumable, incremental)

topochange over the whole (sill=1) dataset; all packages on the stratified `EVAL_SELECTION`. Each runner appends per-file to CSV and skips files already done, so it's safe to interrupt/rerun.

In [ ]:
### Scoring helpers: resumable + incremental fitting (auto / known-model, with param_std)
import os

def _append_csv(path, rows, columns=None):
    """Append rows, always conforming to a fixed header (resume-safe):
    a fresh file gets `columns`; an existing file is matched to ITS header."""
    if not rows:
        return
    df = pd.DataFrame(rows)
    if os.path.exists(path):
        try:
            cols = list(pd.read_csv(path, nrows=0).columns)
        except Exception:
            cols = columns
        (df.reindex(columns=cols) if cols is not None else df).to_csv(path, mode="a", header=False, index=False)
    else:
        (df.reindex(columns=columns) if columns is not None else df).to_csv(path, mode="w", header=True, index=False)

def _done_keys(path, cols):
    if os.path.exists(path):
        try:
            d = pd.read_csv(path, usecols=cols)
            return set(map(tuple, d[cols].astype(str).values))
        except Exception:
            return set()
    return set()

_META = ["file","kind","model","range","sill","nugget_frac","separation_ratio",
         "syserr","slope","ani_ratio","ani_major","n_components"]
def _meta(row):
    return {c: (row[c] if c in row.index else np.nan) for c in _META}

# fixed output schema for the fit runners (guarantees a well-formed CSV under append)
FIT_COLS = _META + ["true_nugget","mode","package","comp","pseudo_r2","rec_nugget",
                    "true_model","true_range","true_sill","rec_model","rec_range","rec_sill",
                    "rec_nu","rec_alpha","range_relerr","sill_err","model_correct",
                    "range_std","sill_std","nugget_std","error"]

def _known_restrict(pkg, family):
    """kwargs restricting a package to `family` (None -> package cannot fit that family)."""
    fam = str(family).lower(); Fam = str(family).capitalize()
    if pkg == "topochange":
        return {"model_types": [fam]} if fam in ("spherical","exponential","matern") else None
    if pkg == "gstools":
        return {"models": {Fam: getattr(gs, Fam)}} if hasattr(gs, Fam) else None
    if pkg in ("scikit-gstat","scikit-gstat-native"):
        return {"models": [fam]} if fam in skg_models else None
    return None

def _fit_row(row, package_fns, mode):
    """One raster x packages in `mode` ('auto'|'known') -> long scored rows.
    n_components==1 -> single fitters (+known restriction); >1 -> nested fitters (auto only)."""
    path = row["path"]; truth = truth_from_tags(path)
    tcomps = sorted(_truth_components(truth),
                    key=lambda c: (c["range"] if np.isfinite(c["range"]) else 1e18))
    k = len(tcomps)
    if k == 0:
        return []
    base = {**_meta(row), "true_nugget": truth["nugget"], "mode": mode}
    rows = []
    for pkg, fn in package_fns.items():
        try:
            if k == 1:
                if mode == "known":
                    restr = _known_restrict(pkg, tcomps[0]["model"])
                    if restr is None:
                        continue
                    res = fn(path, **restr)
                else:
                    res = fn(path)
                rcomps = _as_components(res, "single")
            else:
                if mode == "known":            # nested fitters are already spherical-family
                    continue
                res = fn(path, k); rcomps = _as_components(res, "nested")
            rnug = res.get("nugget", np.nan); r2 = res.get("pseudo_r2", np.nan)
        except Exception as e:
            rows.append({**base, "package": pkg, "comp": np.nan, "error": str(e)[:120]}); continue
        for i, tc in enumerate(tcomps):
            rc = rcomps[i] if i < len(rcomps) else None
            r = {**base, "package": pkg, "comp": i+1, "pseudo_r2": r2, "rec_nugget": rnug,
                 "true_model": tc["model"], "true_range": tc["range"], "true_sill": tc["partial_sill"],
                 "rec_model": (rc["model"] if rc else None),
                 "rec_range": (rc["range"] if rc else np.nan),
                 "rec_sill": (rc["partial_sill"] if rc else np.nan),
                 "rec_nu": (rc.get("nu") if rc else np.nan),
                 "rec_alpha": (rc.get("alpha") if rc else np.nan),
                 "range_relerr": ((rc["range"]-tc["range"])/tc["range"] if (rc and tc["range"]) else np.nan),
                 "sill_err": (rc["partial_sill"]-tc["partial_sill"] if rc else np.nan),
                 "model_correct": (rc is not None and str(rc["model"]).lower()==str(tc["model"]).lower())}
            if i == 0:                          # param uncertainty (single-fit only)
                r.update(range_std=res.get("range_std", np.nan),
                         sill_std=res.get("var_std", np.nan),
                         nugget_std=res.get("nugget_std", np.nan))
            rows.append(r)
    return rows

def run_fits_incremental(selection, package_fns, out_csv, modes=("auto","known"), progress=100):
    """Resumable per-file fitting; appends rows to out_csv as it goes (skips files already present)."""
    done = {t[0] for t in _done_keys(out_csv, ["file"])}
    todo = selection[~selection["file"].astype(str).isin(done)]
    print(f"{len(done)} done, {len(todo)} to do -> {out_csv}")
    for i, (_, row) in enumerate(todo.iterrows()):
        rows = []
        for mode in modes:
            rows += _fit_row(row, package_fns, mode)
        _append_csv(out_csv, rows, columns=FIT_COLS)
        if progress and (i+1) % progress == 0:
            print(f"  [{i+1}/{len(todo)}] {row['file']}")
    print("done ->", out_csv)

In [ ]:
### Fit topochange over the whole dataset (sill=1) -- resumable, incremental
TOPO_SELECTION = catalog[((catalog.sill == 1.0) | (catalog.sill.isna())) & (catalog.kind != "nugget")].reset_index(drop=True)
print(f"topochange full-fit selection: {len(TOPO_SELECTION)} files (sill=1, non-nugget)")

RUN_TOPO_ALL = False    # <-- set True; interruptible & resumable (appends per file, skips done)
if RUN_TOPO_ALL:
    run_fits_incremental(TOPO_SELECTION, {"topochange": fit_topochange},
                         os.path.join(DATA_DIR, "topochange_fits.csv"))

In [ ]:
### Inter-package fitting on the stratified subsample -- resumable, incremental (auto+known, param_std)
INTERPKG_FITTERS = {"topochange": fit_topochange, "gstools": fit_gstools,
                    "scikit-gstat": fit_skg, "scikit-gstat-native": fit_skg_native}

RUN_INTERPKG = True    # <-- set True
if RUN_INTERPKG:
    run_fits_incremental(EVAL_SELECTION, INTERPKG_FITTERS,
                         os.path.join(DATA_DIR, "interpackage_fits.csv"))

In [ ]:
### topochange parameter uncertainty via bootstrap -- subset (default 200), resumable
UNC_N = 200             # raise to len(EVAL single-comp) to run all
UNC_SELECTION = EVAL_SELECTION[EVAL_SELECTION.n_components == 1].head(UNC_N)
print(f"bootstrap uncertainty subset: {len(UNC_SELECTION)} single-component files")

def _gkey(nm):                                  # map a param name -> generic {sill,range,nu,nugget}
    for g in ("sill", "range", "nu", "nugget"):
        if nm.endswith(g):
            return g
    return None

BOOT_COLS = _META + ["package","n_boot","best_model"] + \
            [f"boot_{s}__{g}" for g in ("sill","range","nu","nugget") for s in ("std","p16","p84")] + ["error"]

def _boot_row(row, n_boot=200):
    path = row["path"]; rdh = RasterDataHandler(path, unit="m", resolution=1.0); rdh.load_raster()
    sv = SingleVariogram(rdh)
    sv.compute_empirical_variogram(area_side=1.0, samples_per_area=1.0, max_samples=3000, bin_width=2.0,
                                   max_lag_multiplier=1/3, seed=42, estimator="matheron", return_sample=True)
    sv.fit_model(model_types=["spherical","exponential","matern"], include_nugget=True, criterion="aicc", seed=42)
    bm = sv.best_model; names = list(bm["model"].param_names); samp = np.asarray(bm.get("param_samples"))
    fam = bm["model"].component_names[0] if bm["model"].component_names else "nugget"
    out = {**_meta(row), "package": "topochange", "n_boot": n_boot, "best_model": fam}
    if samp is not None and samp.ndim == 2:
        std = samp.std(0); p16, p84 = np.percentile(samp, [16, 84], axis=0)
        for j, nm in enumerate(names):
            gk = _gkey(nm)
            if gk:
                out[f"boot_std__{gk}"] = float(std[j])
                out[f"boot_p16__{gk}"] = float(p16[j]); out[f"boot_p84__{gk}"] = float(p84[j])
    return [out]

RUN_BOOTSTRAP = True   # <-- set True (slow: bootstrap per file); resumable
if RUN_BOOTSTRAP:
    out = os.path.join(DATA_DIR, "bootstrap_unc.csv")
    done = {t[0] for t in _done_keys(out, ["file"])}
    todo = UNC_SELECTION[~UNC_SELECTION["file"].astype(str).isin(done)]
    print(f"{len(done)} done, {len(todo)} to do")
    for i, (_, row) in enumerate(todo.iterrows()):
        try:
            rows = _boot_row(row)
        except Exception as e:
            rows = [{**_meta(row), "package": "topochange", "error": str(e)[:120]}]
        _append_csv(out, rows, columns=BOOT_COLS)
        if (i+1) % 20 == 0: print(f"  [{i+1}/{len(todo)}]")
    print("done ->", out)

### Tier 3 -- areal uncertainty from fitted params (reuse fits)

In [ ]:
### Areal uncertainty from the fitted params (reuse interpackage fits; exact FFT) -- resumable
def run_areal_eval(fits_csv=None, out_csv=None, progress=50):
    fits_csv = fits_csv or os.path.join(DATA_DIR, "interpackage_fits.csv")
    out_csv  = out_csv  or os.path.join(DATA_DIR, "areal_eval.csv")
    F = pd.read_csv(fits_csv); F = F[F["rec_model"].notna()]
    AREAL_COLS = ["file","package","mode","family","sigma2","polygon","sigma_A","error"]
    done = _done_keys(out_csv, ["file","package","mode"])
    grps = list(F.groupby(["file","package","mode"])); n = 0
    print(f"{len(grps)} (file,package,mode) groups; {len(done)} already done")
    for (file, pkg, mode), grp in grps:
        if (str(file), str(pkg), str(mode)) in done:
            continue
        path = os.path.join(DATA_DIR, str(file))
        comps = [{"model": r.rec_model, "range": r.rec_range, "partial_sill": r.rec_sill,
                  "nu": (None if pd.isna(r.rec_nu) else r.rec_nu),
                  "alpha": (None if pd.isna(r.rec_alpha) else r.rec_alpha)}
                 for r in grp.itertuples() if pd.notna(r.rec_range)]
        nug = float(grp["rec_nugget"].iloc[0]) if pd.notna(grp["rec_nugget"].iloc[0]) else 0.0
        try:
            if len(comps) == 1:
                c = comps[0]; rowm = {"method": pkg, "best_model": c["model"], "var": c["partial_sill"],
                                      "sill": c["partial_sill"] + nug, "nugget": nug, "range": c["range"],
                                      "nu": c["nu"], "alpha": c["alpha"]}
            else:
                rowm = {"method": pkg, "components": comps, "nugget": nug}
            m = model_from_row(rowm)
            recs = [{"file": file, "package": pkg, "mode": mode, "family": m["family"], "sigma2": m["sigma2"],
                     "polygon": pn, "sigma_A": unc_fft(m, geom, path)} for pn, geom in poly_geoms.items()]
        except Exception as e:
            recs = [{"file": file, "package": pkg, "mode": mode, "error": str(e)[:120]}]
        _append_csv(out_csv, recs, columns=AREAL_COLS); n += 1
        if progress and n % progress == 0: print(f"  [{n}]")
    print("done ->", out_csv)

RUN_AREAL_EVAL = True   # <-- run AFTER interpackage fits exist
if RUN_AREAL_EVAL:
    run_areal_eval()